# AgroInvest
## 03 - Modelado predictivo

En este notebook se desarrolla el módulo predictivo de AgroInvest.

El objetivo es estimar el beneficio por hectárea asociado a las características de una explotación y a su configuración tecnológica.

Para evitar que las variaciones históricas de precios condicionen el aprendizaje del modelo, se utilizará inicialmente el dataset con precios constantes. De esta forma, el análisis se centra principalmente en las diferencias productivas, operativas y tecnológicas observadas.

Se compararán varios enfoques:

- modelo baseline;
- regresión Ridge;
- Random Forest.

Los modelos se evaluarán mediante MAE, RMSE y R².

Dado que el conjunto de datos contiene observaciones repetidas de las mismas parcelas en diferentes campañas, la separación entre entrenamiento y prueba se realizará por parcela para reducir el riesgo de fuga de información.

Las predicciones se interpretarán como estimaciones basadas en los patrones observados en los datos y no como efectos causales de las tecnologías.

In [15]:
import pandas as pd
import numpy as np
import numpy_financial as npf
from sklearn.model_selection import GroupShuffleSplit

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [16]:
df = pd.read_excel("Balance_economico.xlsx")

In [17]:
#limpieza básica
df.columns = df.columns.str.strip()

df["Parcela"] = df["Parcela"].astype(str).str.strip()
df["Cultivo"] = df["Cultivo"].astype(str).str.strip()

In [18]:
df.shape

(105, 36)

In [19]:
df[["Año", "Parcela", "Cultivo", "Beneficio_por_ha"]].head()

,Año,Parcela,Cultivo,Beneficio_por_ha
0,2020,Cerro Cruz Cati,Maiz,1239.545625
1,2021,Cerro Cruz Cati,Maiz,1762.802326
2,2023,Cerro Cruz Cati,Tomates,3988.970304
3,2025,Cerro Cruz Cati,Maiz,1621.959594
4,2026,Cerro Cruz Cati,Maiz,2455.094446


In [20]:
variables_modelo = [
    "Cultivo",
    "Superficie",

    "GPS",
    "RTK",
    "ISOBUS",
    "Siembra_variable",
    "Abono_variable",
    "Corte_tramos",

    "Rend_t_ha",
    "Productividad_ha_h",
    "Consumo_L_ha",
    "Fitosanitarios_ha",
    "Semillas_ha",
    "Abono_ha",
    "Pasadas_ha"
]

objetivo = "Beneficio_por_ha"

In [21]:
X = df[variables_modelo].copy()
y = df[objetivo].copy()

In [22]:
X.head()

,Cultivo,Superficie,GPS,RTK,ISOBUS,Siembra_variable,Abono_variable,Corte_tramos,Rend_t_ha,Productividad_ha_h,Consumo_L_ha,Fitosanitarios_ha,Semillas_ha,Abono_ha,Pasadas_ha
0,Maiz,15.4,1,0,0,0,0,0,14.50,1.60,12.50,4.25,102000,760,9
1,Maiz,15.4,1,1,1,1,1,1,16.12,1.90,10.20,4.00,100000,700,9
2,Tomates,15.4,1,1,1,0,1,1,95.35,1.30,18.88,13.50,26000,1350,15
3,Maiz,15.4,1,1,1,1,0,1,15.90,1.85,10.54,4.00,100000,760,9
4,Maiz,15.4,1,1,1,1,1,1,18.20,1.90,10.20,4.00,100000,700,9


In [23]:
y.head()

0    1239.545625
1    1762.802326
2    3988.970304
3    1621.959594
4    2455.094446
Name: Beneficio_por_ha, dtype: float64

In [24]:
#separamos parcelas completas
grupos = df["Parcela"]

In [25]:
separador = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    separador.split(X, y, groups=grupos)
)

In [26]:
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [27]:
print("Entrenamiento:", X_train.shape)
print("Test:", X_test.shape)

Entrenamiento: (83, 15)
Test: (22, 15)


In [28]:
#ninguna parcela está en los dos conjuntos
parcelas_train = set(df.iloc[train_idx]["Parcela"])
parcelas_test = set(df.iloc[test_idx]["Parcela"])

parcelas_train.intersection(parcelas_test)

set()

In [29]:
variables_categoricas = [
    "Cultivo"
]

variables_numericas = [
    "Superficie",

    "GPS",
    "RTK",
    "ISOBUS",
    "Siembra_variable",
    "Abono_variable",
    "Corte_tramos",

    "Rend_t_ha",
    "Productividad_ha_h",
    "Consumo_L_ha",
    "Fitosanitarios_ha",
    "Semillas_ha",
    "Abono_ha",
    "Pasadas_ha"
]

In [30]:
preprocesador = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            variables_categoricas
        ),
        (
            "numericas",
            StandardScaler(),
            variables_numericas
        )
    ]
)

In [31]:
#baseline
#predecir siempre el beneficio medio del conjunto de entrenamiento
prediccion_baseline = np.full(
    shape=len(y_test),
    fill_value=y_train.mean()
)

In [32]:
#calculamos métrica
mae_baseline = mean_absolute_error(
    y_test,
    prediccion_baseline
)

rmse_baseline = np.sqrt(
    mean_squared_error(
        y_test,
        prediccion_baseline
    )
)

r2_baseline = r2_score(
    y_test,
    prediccion_baseline
)

print(f"MAE baseline: {mae_baseline:.2f} €/ha")
print(f"RMSE baseline: {rmse_baseline:.2f} €/ha")
print(f"R² baseline: {r2_baseline:.3f}")

MAE baseline: 1098.44 €/ha
RMSE baseline: 1348.96 €/ha
R² baseline: -0.365


In [33]:
#RIDGE
modelo_ridge = Pipeline(
    steps=[
        ("preprocesador", preprocesador),
        ("modelo", Ridge(alpha=1.0))
    ]
)

In [34]:
#entreamos
modelo_ridge.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocesador',
                 ColumnTransformer(transformers=[('categoricas',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Cultivo']),
                                                 ('numericas', StandardScaler(),
                                                  ['Superficie', 'GPS', 'RTK',
                                                   'ISOBUS', 'Siembra_variable',
                                                   'Abono_variable',
                                                   'Corte_tramos', 'Rend_t_ha',
                                                   'Productividad_ha_h',
                                                   'Consumo_L_ha',
                                                   'Fitosanitarios_ha',
                                                   'Semillas_ha', 'Abono_ha',
                                                   'Pasadas_ha'])])),
                ('modelo', Ridge())])

In [35]:
#predicción
pred_ridge = modelo_ridge.predict(
    X_test
)

In [36]:
#métricas
mae_ridge = mean_absolute_error(
    y_test,
    pred_ridge
)

rmse_ridge = np.sqrt(
    mean_squared_error(
        y_test,
        pred_ridge
    )
)

r2_ridge = r2_score(
    y_test,
    pred_ridge
)

print(f"MAE Ridge: {mae_ridge:.2f} €/ha")
print(f"RMSE Ridge: {rmse_ridge:.2f} €/ha")
print(f"R² Ridge: {r2_ridge:.3f}")

MAE Ridge: 135.24 €/ha
RMSE Ridge: 170.77 €/ha
R² Ridge: 0.978


In [37]:
#random forest
modelo_rf = Pipeline(
    steps=[
        (
            "preprocesador",
            preprocesador
        ),
        (
            "modelo",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                min_samples_leaf=2
            )
        )
    ]
)

In [38]:
modelo_rf.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocesador',
                 ColumnTransformer(transformers=[('categoricas',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Cultivo']),
                                                 ('numericas', StandardScaler(),
                                                  ['Superficie', 'GPS', 'RTK',
                                                   'ISOBUS', 'Siembra_variable',
                                                   'Abono_variable',
                                                   'Corte_tramos', 'Rend_t_ha',
                                                   'Productividad_ha_h',
                                                   'Consumo_L_ha',
                                                   'Fitosanitarios_ha',
                                                   'Semillas_ha', 'Abono_ha',
                                                   'Pasadas_ha'])])),
                ('modelo',
                 RandomForestRegressor(min_samples_leaf=2, n_estimators=300,
                                       random_state=42))])

In [39]:
pred_rf = modelo_rf.predict(
    X_test
)

In [40]:
mae_rf = mean_absolute_error(
    y_test,
    pred_rf
)

rmse_rf = np.sqrt(
    mean_squared_error(
        y_test,
        pred_rf
    )
)

r2_rf = r2_score(
    y_test,
    pred_rf
)

print(f"MAE Random Forest: {mae_rf:.2f} €/ha")
print(f"RMSE Random Forest: {rmse_rf:.2f} €/ha")
print(f"R² Random Forest: {r2_rf:.3f}")

MAE Random Forest: 36.47 €/ha
RMSE Random Forest: 50.13 €/ha
R² Random Forest: 0.998


In [41]:
resultados_modelos = pd.DataFrame({
    "Modelo": [
        "Baseline",
        "Ridge",
        "Random Forest"
    ],
    "MAE": [
        mae_baseline,
        mae_ridge,
        mae_rf
    ],
    "RMSE": [
        rmse_baseline,
        rmse_ridge,
        rmse_rf
    ],
    "R2": [
        r2_baseline,
        r2_ridge,
        r2_rf
    ]
})

resultados_modelos.round(3)

,Modelo,MAE,RMSE,R2
0,Baseline,1098.441,1348.958,-0.365
1,Ridge,135.237,170.770,0.978
2,Random Forest,36.474,50.132,0.998


## Modelo predictivo orientado a la decisión

El modelo anterior permite comprobar la capacidad predictiva de las variables productivas y operativas una vez finalizada la campaña. Sin embargo, no es adecuado como motor principal de AgroInvest, ya que utiliza información que no se conoce antes de realizar una inversión.

Por este motivo, se desarrolla un segundo enfoque basado exclusivamente en variables disponibles en el momento de tomar la decisión:

- cultivo;
- superficie;
- tecnologías instaladas.

A partir de estas variables se estimarán diferentes resultados técnicos de la explotación. Posteriormente, estas predicciones se introducirán en el módulo económico de AgroInvest para analizar la rentabilidad de distintas configuraciones tecnológicas.

Este enfoque permite separar el modelo técnico de las condiciones económicas de mercado y facilita la simulación de escenarios futuros.

In [44]:
#variables que conocerá el usuario
variables_decision = [
    "Cultivo",
    "Superficie",
    "GPS",
    "RTK",
    "ISOBUS",
    "Siembra_variable",
    "Abono_variable",
    "Corte_tramos"
]

X_decision = df[variables_decision].copy()

In [45]:
#esperamos predecir
objetivos_tecnicos = [
    "Rend_t_ha",
    "Productividad_ha_h",
    "Consumo_L_ha",
    "Fitosanitarios_ha",
    "Semillas_ha",
    "Abono_ha",
    "Pasadas_ha"
]

In [46]:
X_decision_train = X_decision.iloc[train_idx]
X_decision_test = X_decision.iloc[test_idx]

In [47]:
variables_categoricas_decision = [
    "Cultivo"
]

variables_numericas_decision = [
    "Superficie",
    "GPS",
    "RTK",
    "ISOBUS",
    "Siembra_variable",
    "Abono_variable",
    "Corte_tramos"
]

In [48]:
preprocesador_decision = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            variables_categoricas_decision
        ),
        (
            "numericas",
            StandardScaler(),
            variables_numericas_decision
        )
    ]
)

In [49]:
from sklearn.base import clone

In [50]:
def evaluar_objetivo_tecnico(
    df,
    objetivo,
    X_train,
    X_test,
    train_idx,
    test_idx
):
    
    y_train = df.iloc[train_idx][objetivo]
    y_test = df.iloc[test_idx][objetivo]
    
    # ---------------------------
    # Baseline
    # ---------------------------
    
    pred_baseline = np.full(
        len(y_test),
        y_train.mean()
    )
    
    # ---------------------------
    # Ridge
    # ---------------------------
    
    modelo_ridge = Pipeline(
        steps=[
            (
                "preprocesador",
                clone(preprocesador_decision)
            ),
            (
                "modelo",
                Ridge(alpha=1.0)
            )
        ]
    )
    
    modelo_ridge.fit(
        X_train,
        y_train
    )
    
    pred_ridge = modelo_ridge.predict(
        X_test
    )
    
    # ---------------------------
    # Random Forest
    # ---------------------------
    
    modelo_rf = Pipeline(
        steps=[
            (
                "preprocesador",
                clone(preprocesador_decision)
            ),
            (
                "modelo",
                RandomForestRegressor(
                    n_estimators=300,
                    random_state=42,
                    min_samples_leaf=2
                )
            )
        ]
    )
    
    modelo_rf.fit(
        X_train,
        y_train
    )
    
    pred_rf = modelo_rf.predict(
        X_test
    )
    
    # ---------------------------
    # Métricas
    # ---------------------------
    
    resultados = []
    
    for nombre, pred in [
        ("Baseline", pred_baseline),
        ("Ridge", pred_ridge),
        ("Random Forest", pred_rf)
    ]:
        
        mae = mean_absolute_error(
            y_test,
            pred
        )
        
        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                pred
            )
        )
        
        r2 = r2_score(
            y_test,
            pred
        )
        
        resultados.append({
            "Objetivo": objetivo,
            "Modelo": nombre,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })
    
    return resultados

In [51]:
#evaluamos todos los objetivos
resultados_tecnicos = []

for objetivo in objetivos_tecnicos:
    
    resultados = evaluar_objetivo_tecnico(
        df=df,
        objetivo=objetivo,
        X_train=X_decision_train,
        X_test=X_decision_test,
        train_idx=train_idx,
        test_idx=test_idx
    )
    
    resultados_tecnicos.extend(resultados)

In [52]:
df_resultados_tecnicos = pd.DataFrame(
    resultados_tecnicos
)

df_resultados_tecnicos.round(3)

,Objetivo,Modelo,MAE,RMSE,R2
0,Rend_t_ha,Baseline,17.069,22.160,-0.244
1,Rend_t_ha,Ridge,2.636,3.398,0.971
2,Rend_t_ha,Random Forest,0.909,2.390,0.986
3,Productividad_ha_h,Baseline,0.247,0.270,-0.030
4,Productividad_ha_h,Ridge,0.015,0.017,0.996
5,Productividad_ha_h,Random Forest,0.013,0.018,0.995
6,Consumo_L_ha,Baseline,3.165,3.381,-0.009
7,Consumo_L_ha,Ridge,0.406,0.521,0.976
8,Consumo_L_ha,Random Forest,0.205,0.380,0.987
9,Fitosanitarios_ha,Baseline,1.535,2.118,-0.166


In [53]:
comparacion_r2 = (
    df_resultados_tecnicos
    .pivot(
        index="Objetivo",
        columns="Modelo",
        values="R2"
    )
    .round(3)
)

comparacion_r2

Modelo,Baseline,Random Forest,Ridge
Objetivo,,,
Abono_ha,-0.203,0.999,0.992
Consumo_L_ha,-0.009,0.987,0.976
Fitosanitarios_ha,-0.166,1.000,0.987
Pasadas_ha,-0.087,1.000,0.993
Productividad_ha_h,-0.030,0.995,0.996
Rend_t_ha,-0.244,0.986,0.971
Semillas_ha,-0.059,1.000,0.999


## Validación con configuraciones tecnológicas no observadas

Las métricas anteriores evalúan la capacidad del modelo para generalizar a parcelas no utilizadas durante el entrenamiento. Sin embargo, AgroInvest también pretende evaluar configuraciones tecnológicas que pueden no haberse observado exactamente en los datos históricos.

Por ello, se realiza una segunda validación en la que cada configuración tecnológica se excluye completamente del conjunto de entrenamiento y se utiliza posteriormente como conjunto de prueba.

De esta forma se analiza la capacidad de los modelos para estimar resultados técnicos ante combinaciones tecnológicas no observadas previamente.

In [55]:
#quitamos GPS ya que siempre se ha utilizado
variables_decision_final = [
    "Cultivo",
    "Superficie",
    "RTK",
    "ISOBUS",
    "Siembra_variable",
    "Abono_variable",
    "Corte_tramos"
]

X_final = df[variables_decision_final].copy()

In [56]:
columnas_configuracion = [
    "RTK",
    "ISOBUS",
    "Siembra_variable",
    "Abono_variable",
    "Corte_tramos"
]

In [57]:
df["Configuracion"] = (
    df[columnas_configuracion]
    .astype(str)
    .agg("-".join, axis=1)
)

In [58]:
df[
    ["Configuracion"] + columnas_configuracion
].drop_duplicates().sort_values("Configuracion")

,Configuracion,RTK,ISOBUS,Siembra_variable,Abono_variable,Corte_tramos
0,0-0-0-0-0,0,0,0,0,0
7,1-1-0-1-0,1,1,0,1,0
2,1-1-0-1-1,1,1,0,1,1
3,1-1-1-0-1,1,1,1,0,1
71,1-1-1-1-0,1,1,1,1,0
1,1-1-1-1-1,1,1,1,1,1


In [59]:
df["Configuracion"].value_counts()

Configuracion
1-1-1-1-1    50
1-1-0-1-1    23
0-0-0-0-0    18
1-1-1-0-1     6
1-1-1-1-0     6
1-1-0-1-0     2
Name: count, dtype: int64

In [60]:
from sklearn.model_selection import LeaveOneGroupOut

In [61]:
variables_categoricas_final = [
    "Cultivo"
]

variables_numericas_final = [
    "Superficie",
    "RTK",
    "ISOBUS",
    "Siembra_variable",
    "Abono_variable",
    "Corte_tramos"
]

In [62]:
preprocesador_final = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            variables_categoricas_final
        ),
        (
            "numericas",
            StandardScaler(),
            variables_numericas_final
        )
    ]
)

In [63]:
def validar_configuraciones_no_vistas(
    df,
    X,
    objetivo
):
    
    grupos = df["Configuracion"]
    
    logo = LeaveOneGroupOut()
    
    predicciones_ridge = np.zeros(len(df))
    predicciones_rf = np.zeros(len(df))
    
    for train_idx_config, test_idx_config in logo.split(
        X,
        df[objetivo],
        groups=grupos
    ):
        
        X_train_config = X.iloc[train_idx_config]
        X_test_config = X.iloc[test_idx_config]
        
        y_train_config = df.iloc[train_idx_config][objetivo]
        
        # ------------------------
        # RIDGE
        # ------------------------
        
        ridge = Pipeline(
            steps=[
                (
                    "preprocesador",
                    clone(preprocesador_final)
                ),
                (
                    "modelo",
                    Ridge(alpha=1.0)
                )
            ]
        )
        
        ridge.fit(
            X_train_config,
            y_train_config
        )
        
        predicciones_ridge[test_idx_config] = (
            ridge.predict(X_test_config)
        )
        
        # ------------------------
        # RANDOM FOREST
        # ------------------------
        
        rf = Pipeline(
            steps=[
                (
                    "preprocesador",
                    clone(preprocesador_final)
                ),
                (
                    "modelo",
                    RandomForestRegressor(
                        n_estimators=300,
                        random_state=42,
                        min_samples_leaf=2
                    )
                )
            ]
        )
        
        rf.fit(
            X_train_config,
            y_train_config
        )
        
        predicciones_rf[test_idx_config] = (
            rf.predict(X_test_config)
        )
    
    y_real = df[objetivo]
    
    resultados = []
    
    for nombre, predicciones in [
        ("Ridge", predicciones_ridge),
        ("Random Forest", predicciones_rf)
    ]:
        
        mae = mean_absolute_error(
            y_real,
            predicciones
        )
        
        rmse = np.sqrt(
            mean_squared_error(
                y_real,
                predicciones
            )
        )
        
        r2 = r2_score(
            y_real,
            predicciones
        )
        
        resultados.append({
            "Objetivo": objetivo,
            "Modelo": nombre,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })
    
    return resultados

In [64]:
objetivos_finales = [
    "Rend_t_ha",
    "Productividad_ha_h",
    "Consumo_L_ha",
    "Fitosanitarios_ha",
    "Semillas_ha",
    "Abono_ha",
    "Pasadas_ha"
]

In [65]:
resultados_configuraciones = []

for objetivo in objetivos_finales:
    
    resultado = validar_configuraciones_no_vistas(
        df=df,
        X=X_final,
        objetivo=objetivo
    )
    
    resultados_configuraciones.extend(
        resultado
    )

In [66]:
df_resultados_configuraciones = pd.DataFrame(
    resultados_configuraciones
)

df_resultados_configuraciones.round(3)

,Objetivo,Modelo,MAE,RMSE,R2
0,Rend_t_ha,Ridge,7.055,14.405,0.779
1,Rend_t_ha,Random Forest,7.809,22.202,0.475
2,Productividad_ha_h,Ridge,0.051,0.067,0.948
3,Productividad_ha_h,Random Forest,0.080,0.114,0.852
4,Consumo_L_ha,Ridge,1.145,1.825,0.796
5,Consumo_L_ha,Random Forest,1.093,1.756,0.811
6,Fitosanitarios_ha,Ridge,0.553,0.792,0.934
7,Fitosanitarios_ha,Random Forest,0.604,1.731,0.683
8,Semillas_ha,Ridge,4001.357,4837.021,0.990
9,Semillas_ha,Random Forest,832.124,1861.149,0.999


In [67]:
comparacion_configuraciones = (
    df_resultados_configuraciones
    .pivot(
        index="Objetivo",
        columns="Modelo",
        values="R2"
    )
    .round(3)
)

comparacion_configuraciones

Modelo,Random Forest,Ridge
Objetivo,,
Abono_ha,0.697,0.953
Consumo_L_ha,0.811,0.796
Fitosanitarios_ha,0.683,0.934
Pasadas_ha,0.787,0.969
Productividad_ha_h,0.852,0.948
Rend_t_ha,0.475,0.779
Semillas_ha,0.999,0.990


In [68]:
def evaluar_por_configuracion(
    df,
    X,
    objetivo
):
    
    configuraciones = df["Configuracion"].unique()
    
    resultados = []
    
    for configuracion_test in configuraciones:
        
        mascara_test = (
            df["Configuracion"] == configuracion_test
        )
        
        mascara_train = ~mascara_test
        
        X_train = X.loc[mascara_train]
        X_test = X.loc[mascara_test]
        
        y_train = df.loc[
            mascara_train,
            objetivo
        ]
        
        y_test = df.loc[
            mascara_test,
            objetivo
        ]
        
        modelo = Pipeline(
            steps=[
                (
                    "preprocesador",
                    clone(preprocesador_final)
                ),
                (
                    "modelo",
                    Ridge(alpha=1.0)
                )
            ]
        )
        
        modelo.fit(
            X_train,
            y_train
        )
        
        pred = modelo.predict(
            X_test
        )
        
        mae = mean_absolute_error(
            y_test,
            pred
        )
        
        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                pred
            )
        )
        
        resultados.append({
            "Objetivo": objetivo,
            "Configuracion": configuracion_test,
            "Observaciones": len(y_test),
            "MAE": mae,
            "RMSE": rmse
        })
    
    return resultados

In [69]:
resultados_por_configuracion = []

for objetivo in objetivos_finales:
    
    resultado = evaluar_por_configuracion(
        df=df,
        X=X_final,
        objetivo=objetivo
    )
    
    resultados_por_configuracion.extend(
        resultado
    )

In [70]:
df_por_configuracion = pd.DataFrame(
    resultados_por_configuracion
)

df_por_configuracion.round(3)

,Objetivo,Configuracion,Observaciones,MAE,RMSE
0,Rend_t_ha,0-0-0-0-0,18,4.251,5.624
1,Rend_t_ha,1-1-1-1-1,50,3.701,3.954
2,Rend_t_ha,1-1-0-1-1,23,18.580,29.609
3,Rend_t_ha,1-1-1-0-1,6,1.060,1.297
4,Rend_t_ha,1-1-0-1-0,2,6.054,6.144
5,Rend_t_ha,1-1-1-1-0,6,5.570,5.579
6,Productividad_ha_h,0-0-0-0-0,18,0.102,0.119
7,Productividad_ha_h,1-1-1-1-1,50,0.051,0.059
8,Productividad_ha_h,1-1-0-1-1,23,0.027,0.036
9,Productividad_ha_h,1-1-1-0-1,6,0.020,0.023


In [71]:
df_por_configuracion[
    df_por_configuracion["Objetivo"] == "Rend_t_ha"
].sort_values(
    "MAE",
    ascending=False
).round(3)

,Objetivo,Configuracion,Observaciones,MAE,RMSE
2,Rend_t_ha,1-1-0-1-1,23,18.580,29.609
4,Rend_t_ha,1-1-0-1-0,2,6.054,6.144
5,Rend_t_ha,1-1-1-1-0,6,5.570,5.579
0,Rend_t_ha,0-0-0-0-0,18,4.251,5.624
1,Rend_t_ha,1-1-1-1-1,50,3.701,3.954
3,Rend_t_ha,1-1-1-0-1,6,1.060,1.297


In [72]:
df_por_configuracion[
    df_por_configuracion["Objetivo"] == "Consumo_L_ha"
].sort_values(
    "MAE",
    ascending=False
).round(3)

,Objetivo,Configuracion,Observaciones,MAE,RMSE
12,Consumo_L_ha,0-0-0-0-0,18,2.265,3.190
14,Consumo_L_ha,1-1-0-1-1,23,1.749,2.371
17,Consumo_L_ha,1-1-1-1-0,6,0.732,0.732
16,Consumo_L_ha,1-1-0-1-0,2,0.703,0.704
13,Consumo_L_ha,1-1-1-1-1,50,0.600,0.788
15,Consumo_L_ha,1-1-1-0-1,6,0.574,0.582


In [73]:
# Distribución de configuraciones por cultivo

configuraciones_por_cultivo = (
    df.groupby(
        ["Cultivo", "Configuracion"]
    )
    .size()
    .reset_index(name="Observaciones")
)

configuraciones_por_cultivo

,Cultivo,Configuracion,Observaciones
0,Arroz,0-0-0-0-0,7
1,Arroz,1-1-0-1-1,6
2,Arroz,1-1-1-0-1,3
3,Arroz,1-1-1-1-0,6
4,Arroz,1-1-1-1-1,20
5,Maiz,0-0-0-0-0,9
6,Maiz,1-1-0-1-0,2
7,Maiz,1-1-0-1-1,7
8,Maiz,1-1-1-0-1,3
9,Maiz,1-1-1-1-1,30


In [74]:
# Rendimiento medio de cada combinación cultivo-configuración

rendimiento_configuracion = (
    df.groupby(
        ["Cultivo", "Configuracion"]
    )
    .agg(
        Observaciones=("Rend_t_ha", "size"),
        Rendimiento_medio=("Rend_t_ha", "mean"),
        Rendimiento_min=("Rend_t_ha", "min"),
        Rendimiento_max=("Rend_t_ha", "max")
    )
    .round(2)
    .reset_index()
)

rendimiento_configuracion

,Cultivo,Configuracion,Observaciones,Rendimiento_medio,Rendimiento_min,Rendimiento_max
0,Arroz,0-0-0-0-0,7,7.20,6.90,7.45
1,Arroz,1-1-0-1-1,6,8.14,8.00,8.40
2,Arroz,1-1-1-0-1,3,8.47,8.02,9.20
3,Arroz,1-1-1-1-0,6,8.06,7.88,8.30
4,Arroz,1-1-1-1-1,20,8.53,8.10,9.30
5,Maiz,0-0-0-0-0,9,13.43,12.70,14.50
6,Maiz,1-1-0-1-0,2,14.42,13.53,15.32
7,Maiz,1-1-0-1-1,7,16.02,15.60,16.34
8,Maiz,1-1-1-0-1,3,15.65,15.23,15.90
9,Maiz,1-1-1-1-1,30,16.96,15.44,19.00


In [75]:
df[
    df["Configuracion"] == "1-1-0-1-1"
][
    [
        "Cultivo",
        "Año",
        "Parcela",
        "Rend_t_ha",
        "Productividad_ha_h",
        "Consumo_L_ha"
    ]
]

,Cultivo,Año,Parcela,Rend_t_ha,Productividad_ha_h,Consumo_L_ha
2,Tomates,2023,Cerro Cruz Cati,95.35,1.30,18.880000
8,Tomates,2024,Tomillar,130.00,1.30,18.880000
10,Tomates,2021,Viñas,100.10,1.30,18.880000
11,Maiz,2023,Viñas,16.23,1.85,10.540000
17,Maiz,2023,Viñas Cati,16.34,1.85,10.540000
29,Tomates,2026,Alonso de Ojeda,105.00,1.30,18.880000
31,Maiz,2023,Cerro Cruz Caseta,16.18,1.85,10.540000
32,Tomates,2024,Cerro Cruz Caseta,132.00,1.30,18.880000
36,Tomates,2021,Ventoso,95.00,1.30,18.880000
37,Maiz,2023,Ventoso,16.00,1.85,10.540000


In [76]:
## Modelos específicos por cultivo

In [77]:
df_maiz = df[df["Cultivo"] == "Maiz"].copy()
df_arroz = df[df["Cultivo"] == "Arroz"].copy()
df_tomate = df[df["Cultivo"] == "Tomates"].copy()

In [78]:
variables_por_cultivo = [
    "Superficie",
    "RTK",
    "ISOBUS",
    "Siembra_variable",
    "Abono_variable",
    "Corte_tramos"
]

In [79]:
objetivos_finales = [
    "Rend_t_ha",
    "Productividad_ha_h",
    "Consumo_L_ha",
    "Fitosanitarios_ha",
    "Semillas_ha",
    "Abono_ha",
    "Pasadas_ha"
]

In [80]:
# Datos de entrada para maíz
X_maiz = df_maiz[variables_por_cultivo].copy()

In [81]:
columnas_configuracion_maiz = [
    "RTK",
    "ISOBUS",
    "Siembra_variable",
    "Abono_variable",
    "Corte_tramos"
]

df_maiz["Configuracion"] = (
    df_maiz[columnas_configuracion_maiz]
    .astype(str)
    .agg("-".join, axis=1)
)

In [82]:
df_maiz[
    ["Configuracion"] + columnas_configuracion_maiz
].drop_duplicates().sort_values("Configuracion")

,Configuracion,RTK,ISOBUS,Siembra_variable,Abono_variable,Corte_tramos
0,0-0-0-0-0,0,0,0,0,0
7,1-1-0-1-0,1,1,0,1,0
11,1-1-0-1-1,1,1,0,1,1
3,1-1-1-0-1,1,1,1,0,1
1,1-1-1-1-1,1,1,1,1,1


In [83]:
df_maiz["Configuracion"].value_counts()

Configuracion
1-1-1-1-1    30
0-0-0-0-0     9
1-1-0-1-1     7
1-1-1-0-1     3
1-1-0-1-0     2
Name: count, dtype: int64

In [84]:
preprocesador_maiz = ColumnTransformer(
    transformers=[
        (
            "numericas",
            StandardScaler(),
            variables_por_cultivo
        )
    ]
)

In [85]:
def validar_maiz_configuraciones_no_vistas(
    df,
    X,
    objetivo
):
    
    grupos = df["Configuracion"]
    logo = LeaveOneGroupOut()

    predicciones = np.zeros(len(df))

    for train_idx, test_idx in logo.split(
        X,
        df[objetivo],
        groups=grupos
    ):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = df.iloc[train_idx][objetivo]

        modelo = Pipeline(
            steps=[
                (
                    "preprocesador",
                    clone(preprocesador_maiz)
                ),
                (
                    "modelo",
                    Ridge(alpha=1.0)
                )
            ]
        )

        modelo.fit(
            X_train,
            y_train
        )

        predicciones[test_idx] = modelo.predict(
            X_test
        )

    y_real = df[objetivo]

    mae = mean_absolute_error(
        y_real,
        predicciones
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_real,
            predicciones
        )
    )

    r2 = r2_score(
        y_real,
        predicciones
    )

    return {
        "Objetivo": objetivo,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

In [86]:
resultados_maiz = []

for objetivo in objetivos_finales:

    resultado = validar_maiz_configuraciones_no_vistas(
        df=df_maiz,
        X=X_maiz,
        objetivo=objetivo
    )

    resultados_maiz.append(resultado)

In [87]:
df_resultados_maiz = pd.DataFrame(
    resultados_maiz
)

df_resultados_maiz.round(3)

,Objetivo,MAE,RMSE,R2
0,Rend_t_ha,0.831,1.167,0.439
1,Productividad_ha_h,0.030,0.056,0.744
2,Consumo_L_ha,0.221,0.442,0.733
3,Fitosanitarios_ha,0.026,0.056,0.708
4,Semillas_ha,470.021,570.621,0.644
5,Abono_ha,20.166,24.766,0.053
6,Pasadas_ha,0.000,0.000,1.000


In [88]:
def errores_maiz_por_configuracion(
    df,
    X,
    objetivo
):

    resultados = []

    for configuracion in df["Configuracion"].unique():

        mascara_test = (
            df["Configuracion"] == configuracion
        )

        mascara_train = ~mascara_test

        X_train = X.loc[mascara_train]
        X_test = X.loc[mascara_test]

        y_train = df.loc[
            mascara_train,
            objetivo
        ]

        y_test = df.loc[
            mascara_test,
            objetivo
        ]

        modelo = Pipeline(
            steps=[
                (
                    "preprocesador",
                    clone(preprocesador_maiz)
                ),
                (
                    "modelo",
                    Ridge(alpha=1.0)
                )
            ]
        )

        modelo.fit(
            X_train,
            y_train
        )

        pred = modelo.predict(
            X_test
        )

        resultados.append({
            "Configuracion": configuracion,
            "Observaciones": len(y_test),
            "MAE": mean_absolute_error(
                y_test,
                pred
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    y_test,
                    pred
                )
            )
        })

    return pd.DataFrame(resultados)

In [89]:
errores_rendimiento_maiz = errores_maiz_por_configuracion(
    df=df_maiz,
    X=X_maiz,
    objetivo="Rend_t_ha"
)

errores_rendimiento_maiz.sort_values(
    "MAE",
    ascending=False
).round(3)

,Configuracion,Observaciones,MAE,RMSE
3,1-1-0-1-0,2,1.153,1.451
1,1-1-1-1-1,30,1.049,1.393
2,1-1-1-0-1,3,0.986,1.019
0,0-0-0-0-0,9,0.393,0.560
4,1-1-0-1-1,7,0.303,0.380


## Modelo global con interacciones cultivo-tecnología

Los modelos independientes por cultivo reducen considerablemente el número de observaciones disponibles para el entrenamiento y, en determinadas variables, presentan una capacidad de generalización limitada.

Como alternativa, se plantea un modelo global que utiliza las 105 observaciones disponibles, pero incorpora interacciones entre el cultivo y cada tecnología.

De esta forma, el modelo puede aprender que una misma tecnología puede estar asociada a efectos diferentes en maíz, arroz y tomate, manteniendo al mismo tiempo una mayor cantidad de información durante el entrenamiento.

In [91]:
def crear_variables_interaccion(df):
    
    X = df[
        [
            "Superficie",
            "RTK",
            "ISOBUS",
            "Siembra_variable",
            "Abono_variable",
            "Corte_tramos"
        ]
    ].copy()

    # Variables binarias para cada cultivo
    X["Cultivo_Maiz"] = (df["Cultivo"] == "Maiz").astype(int)
    X["Cultivo_Arroz"] = (df["Cultivo"] == "Arroz").astype(int)
    X["Cultivo_Tomates"] = (df["Cultivo"] == "Tomates").astype(int)

    tecnologias = [
        "RTK",
        "ISOBUS",
        "Siembra_variable",
        "Abono_variable",
        "Corte_tramos"
    ]

    cultivos = [
        "Maiz",
        "Arroz",
        "Tomates"
    ]

    # Interacciones tecnología × cultivo
    for cultivo in cultivos:

        indicador_cultivo = (
            df["Cultivo"] == cultivo
        ).astype(int)

        for tecnologia in tecnologias:

            X[f"{tecnologia}_{cultivo}"] = (
                df[tecnologia] * indicador_cultivo
            )

    return X

In [92]:
X_interacciones = crear_variables_interaccion(df)

X_interacciones.head()

,Superficie,RTK,ISOBUS,Siembra_variable,Abono_variable,Corte_tramos,Cultivo_Maiz,Cultivo_Arroz,Cultivo_Tomates,RTK_Maiz,...,RTK_Arroz,ISOBUS_Arroz,Siembra_variable_Arroz,Abono_variable_Arroz,Corte_tramos_Arroz,RTK_Tomates,ISOBUS_Tomates,Siembra_variable_Tomates,Abono_variable_Tomates,Corte_tramos_Tomates
0,15.4,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,15.4,1,1,1,1,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,15.4,1,1,0,1,1,0,0,1,0,...,0,0,0,0,0,1,1,0,1,1
3,15.4,1,1,1,0,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,15.4,1,1,1,1,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [93]:
preprocesador_interacciones = Pipeline(
    steps=[
        (
            "escalado",
            StandardScaler()
        )
    ]
)

In [94]:
#validación con configuraciones no vistas
def validar_ridge_interacciones(
    df,
    X,
    objetivo
):

    grupos = df["Configuracion"]

    logo = LeaveOneGroupOut()

    predicciones = np.zeros(len(df))

    for train_idx, test_idx in logo.split(
        X,
        df[objetivo],
        groups=grupos
    ):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = df.iloc[train_idx][objetivo]

        modelo = Pipeline(
            steps=[
                (
                    "escalado",
                    StandardScaler()
                ),
                (
                    "modelo",
                    Ridge(alpha=1.0)
                )
            ]
        )

        modelo.fit(
            X_train,
            y_train
        )

        predicciones[test_idx] = modelo.predict(
            X_test
        )

    y_real = df[objetivo]

    return {
        "Objetivo": objetivo,

        "MAE": mean_absolute_error(
            y_real,
            predicciones
        ),

        "RMSE": np.sqrt(
            mean_squared_error(
                y_real,
                predicciones
            )
        ),

        "R2": r2_score(
            y_real,
            predicciones
        )
    }

In [95]:
resultados_interacciones = []

for objetivo in objetivos_finales:

    resultado = validar_ridge_interacciones(
        df=df,
        X=X_interacciones,
        objetivo=objetivo
    )

    resultados_interacciones.append(
        resultado
    )

In [96]:
df_resultados_interacciones = pd.DataFrame(
    resultados_interacciones
)

df_resultados_interacciones.round(3)

,Objetivo,MAE,RMSE,R2
0,Rend_t_ha,4.704,11.284,0.864
1,Productividad_ha_h,0.074,0.120,0.836
2,Consumo_L_ha,1.490,3.028,0.437
3,Fitosanitarios_ha,0.430,1.180,0.853
4,Semillas_ha,6235.162,13995.028,0.915
5,Abono_ha,37.796,85.880,0.842
6,Pasadas_ha,0.252,0.777,0.829


## Entrenamiento final de los modelos predictivos

Tras comparar distintos enfoques de modelado, se selecciona una arquitectura basada principalmente en regresión Ridge.

La validación realizada sobre configuraciones tecnológicas no observadas mostró que Ridge presenta una mejor capacidad de generalización que Random Forest en la mayoría de las variables técnicas.

Además, se comprobó que el rendimiento responde de forma más específica al tipo de cultivo. Por este motivo, para `Rend_t_ha` se utiliza un modelo Ridge con interacciones entre cultivo y tecnología.

Para el resto de variables técnicas se emplean modelos Ridge globales, ya que ofrecieron mejores resultados de validación que los modelos específicos por cultivo o con interacciones.

Los modelos finales se entrenan utilizando las 105 observaciones disponibles, una vez completada la fase de validación. Estos modelos constituirán el motor predictivo de AgroInvest y permitirán estimar resultados técnicos para distintas configuraciones tecnológicas.

In [98]:
#modelo final de rendimiento
# Modelo final para rendimiento
modelo_final_rendimiento = Pipeline(
    steps=[
        (
            "escalado",
            StandardScaler()
        ),
        (
            "modelo",
            Ridge(alpha=1.0)
        )
    ]
)

modelo_final_rendimiento.fit(
    X_interacciones,
    df["Rend_t_ha"]
)

Pipeline(steps=[('escalado', StandardScaler()), ('modelo', Ridge())])

In [99]:
variables_globales_finales = [
    "Cultivo",
    "Superficie",
    "RTK",
    "ISOBUS",
    "Siembra_variable",
    "Abono_variable",
    "Corte_tramos"
]

X_global_final = df[
    variables_globales_finales
].copy()

In [100]:
preprocesador_global_final = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            ["Cultivo"]
        ),
        (
            "numericas",
            StandardScaler(),
            [
                "Superficie",
                "RTK",
                "ISOBUS",
                "Siembra_variable",
                "Abono_variable",
                "Corte_tramos"
            ]
        )
    ]
)

In [101]:
#variables a predecir con ridge global
objetivos_globales_finales = [
    "Productividad_ha_h",
    "Consumo_L_ha",
    "Fitosanitarios_ha",
    "Semillas_ha",
    "Abono_ha",
    "Pasadas_ha"
]

In [102]:
modelos_finales = {}

In [103]:
for objetivo in objetivos_globales_finales:

    modelo = Pipeline(
        steps=[
            (
                "preprocesador",
                clone(preprocesador_global_final)
            ),
            (
                "modelo",
                Ridge(alpha=1.0)
            )
        ]
    )

    modelo.fit(
        X_global_final,
        df[objetivo]
    )

    modelos_finales[objetivo] = modelo

In [104]:
#añadimos tambien el modelo de rendimiento
modelos_finales["Rend_t_ha"] = modelo_final_rendimiento

In [105]:
modelos_finales.keys()

dict_keys(['Productividad_ha_h', 'Consumo_L_ha', 'Fitosanitarios_ha', 'Semillas_ha', 'Abono_ha', 'Pasadas_ha', 'Rend_t_ha'])

In [106]:
#función final de predicción
def predecir_resultados(
    cultivo,
    superficie,
    RTK,
    ISOBUS,
    Siembra_variable,
    Abono_variable,
    Corte_tramos
):
    
    # ----------------------------------
    # Datos para modelos globales
    # ----------------------------------

    entrada_global = pd.DataFrame({
        "Cultivo": [cultivo],
        "Superficie": [superficie],
        "RTK": [RTK],
        "ISOBUS": [ISOBUS],
        "Siembra_variable": [Siembra_variable],
        "Abono_variable": [Abono_variable],
        "Corte_tramos": [Corte_tramos]
    })

    resultados = {}

    # ----------------------------------
    # Predicciones globales
    # ----------------------------------

    for objetivo in objetivos_globales_finales:

        prediccion = modelos_finales[
            objetivo
        ].predict(
            entrada_global
        )[0]

        resultados[objetivo] = prediccion

    # ----------------------------------
    # Entrada para rendimiento
    # ----------------------------------

    entrada_base = pd.DataFrame({
        "Cultivo": [cultivo],
        "Superficie": [superficie],
        "RTK": [RTK],
        "ISOBUS": [ISOBUS],
        "Siembra_variable": [Siembra_variable],
        "Abono_variable": [Abono_variable],
        "Corte_tramos": [Corte_tramos]
    })

    entrada_rendimiento = crear_variables_interaccion(
        entrada_base
    )

    # Aseguramos mismo orden de columnas
    entrada_rendimiento = entrada_rendimiento[
        X_interacciones.columns
    ]

    resultados["Rend_t_ha"] = (
        modelo_final_rendimiento
        .predict(
            entrada_rendimiento
        )[0]
    )

    return resultados

In [107]:
#prueba
resultado_prueba = predecir_resultados(
    cultivo="Maiz",
    superficie=100,
    RTK=1,
    ISOBUS=1,
    Siembra_variable=0,
    Abono_variable=1,
    Corte_tramos=1
)

resultado_prueba

{'Productividad_ha_h': 1.8797657697447523,
 'Consumo_L_ha': 9.164881192288759,
 'Fitosanitarios_ha': 3.774856259703412,
 'Semillas_ha': 105117.4542232815,
 'Abono_ha': 687.3817842510972,
 'Pasadas_ha': 8.79483114243768,
 'Rend_t_ha': 10.054118994637811}

In [108]:
pd.DataFrame(
    resultado_prueba.items(),
    columns=["Variable", "Prediccion"]
).round(2)

,Variable,Prediccion
0,Productividad_ha_h,1.88
1,Consumo_L_ha,9.16
2,Fitosanitarios_ha,3.77
3,Semillas_ha,105117.45
4,Abono_ha,687.38
5,Pasadas_ha,8.79
6,Rend_t_ha,10.05


### Ajuste de las variables de entrada

Durante la prueba del motor predictivo se observó que utilizar la superficie total de la explotación como variable explicativa podía generar extrapolaciones poco realistas.

Las variables objetivo del modelo están expresadas principalmente por hectárea (`t/ha`, `L/ha`, etc.), mientras que la superficie introducida posteriormente por el usuario representa la dimensión total de la explotación sobre la que se desea evaluar la inversión.

Por este motivo, la superficie se excluye del modelo técnico final. Se utilizará posteriormente en el módulo económico para transformar los resultados por hectárea en resultados totales de la explotación.

In [110]:
def crear_variables_interaccion_final(df):
    
    X = df[
        [
            "RTK",
            "ISOBUS",
            "Siembra_variable",
            "Abono_variable",
            "Corte_tramos"
        ]
    ].copy()

    X["Cultivo_Maiz"] = (
        df["Cultivo"] == "Maiz"
    ).astype(int)

    X["Cultivo_Arroz"] = (
        df["Cultivo"] == "Arroz"
    ).astype(int)

    X["Cultivo_Tomates"] = (
        df["Cultivo"] == "Tomates"
    ).astype(int)

    tecnologias = [
        "RTK",
        "ISOBUS",
        "Siembra_variable",
        "Abono_variable",
        "Corte_tramos"
    ]

    cultivos = [
        "Maiz",
        "Arroz",
        "Tomates"
    ]

    for cultivo in cultivos:

        indicador = (
            df["Cultivo"] == cultivo
        ).astype(int)

        for tecnologia in tecnologias:

            X[f"{tecnologia}_{cultivo}"] = (
                df[tecnologia] * indicador
            )

    return X

In [111]:
X_interacciones_final = crear_variables_interaccion_final(df)

In [112]:
modelo_final_rendimiento = Pipeline(
    steps=[
        ("escalado", StandardScaler()),
        ("modelo", Ridge(alpha=1.0))
    ]
)

modelo_final_rendimiento.fit(
    X_interacciones_final,
    df["Rend_t_ha"]
)

Pipeline(steps=[('escalado', StandardScaler()), ('modelo', Ridge())])

In [113]:
variables_globales_finales = [
    "Cultivo",
    "RTK",
    "ISOBUS",
    "Siembra_variable",
    "Abono_variable",
    "Corte_tramos"
]

X_global_final = df[
    variables_globales_finales
].copy()

In [114]:
preprocesador_global_final = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            ["Cultivo"]
        ),
        (
            "numericas",
            StandardScaler(),
            [
                "RTK",
                "ISOBUS",
                "Siembra_variable",
                "Abono_variable",
                "Corte_tramos"
            ]
        )
    ]
)

In [115]:
modelos_finales = {}

for objetivo in objetivos_globales_finales:

    modelo = Pipeline(
        steps=[
            (
                "preprocesador",
                clone(preprocesador_global_final)
            ),
            (
                "modelo",
                Ridge(alpha=1.0)
            )
        ]
    )

    modelo.fit(
        X_global_final,
        df[objetivo]
    )

    modelos_finales[objetivo] = modelo

modelos_finales["Rend_t_ha"] = modelo_final_rendimiento

In [116]:
def predecir_resultados(
    cultivo,
    RTK,
    ISOBUS,
    Siembra_variable,
    Abono_variable,
    Corte_tramos
):

    entrada_global = pd.DataFrame({
        "Cultivo": [cultivo],
        "RTK": [RTK],
        "ISOBUS": [ISOBUS],
        "Siembra_variable": [Siembra_variable],
        "Abono_variable": [Abono_variable],
        "Corte_tramos": [Corte_tramos]
    })

    resultados = {}

    for objetivo in objetivos_globales_finales:

        resultados[objetivo] = (
            modelos_finales[objetivo]
            .predict(entrada_global)[0]
        )

    entrada_rendimiento = (
        crear_variables_interaccion_final(
            entrada_global
        )
    )

    entrada_rendimiento = entrada_rendimiento[
        X_interacciones_final.columns
    ]

    resultados["Rend_t_ha"] = (
        modelo_final_rendimiento
        .predict(entrada_rendimiento)[0]
    )

    return resultados

In [117]:
resultado_prueba = predecir_resultados(
    cultivo="Maiz",
    RTK=1,
    ISOBUS=1,
    Siembra_variable=0,
    Abono_variable=1,
    Corte_tramos=1
)

pd.DataFrame(
    resultado_prueba.items(),
    columns=["Variable", "Prediccion"]
).round(2)

,Variable,Prediccion
0,Productividad_ha_h,1.84
1,Consumo_L_ha,10.17
2,Fitosanitarios_ha,4.12
3,Semillas_ha,99816.17
4,Abono_ha,714.26
5,Pasadas_ha,9.09
6,Rend_t_ha,16.04


## Conexión entre el modelo técnico y el motor económico

Una vez entrenados los modelos predictivos, sus resultados se utilizan como entrada del módulo económico de AgroInvest.

El modelo técnico estima variables expresadas principalmente por hectárea, como rendimiento, consumo de combustible, uso de fitosanitarios, semillas y fertilizante. Posteriormente, estas estimaciones se combinan con los precios y costes del escenario económico seleccionado.

En una primera fase se utiliza el escenario de precios constantes, lo que permite analizar las diferencias económicas asociadas a las configuraciones tecnológicas reduciendo el efecto de la volatilidad del mercado.

La superficie no interviene en la predicción técnica, sino que se incorpora posteriormente para transformar los resultados económicos por hectárea en resultados totales de la explotación.

In [120]:
parametros_constantes = (
    df.groupby("Cultivo")
    .agg(
        Precio_tonelada_venta=("Precio_tonelada_venta", "mean"),
        Precio_litro_gasoleoB=("Precio_litro_gasoleoB", "mean"),
        Precio_litro_Fitosanitario=("Precio_litro_Fitosanitario", "mean"),
        Precio_Abono_kg=("Precio_Abono_kg", "mean"),
        Gastos_fijos_por_ha=("Gastos_fijos_por_ha", "mean"),
        Descuento_humedad_por_t=("Descuento_humedad_por_t", "mean")
    )
    .round(4)
)

parametros_constantes

,Precio_tonelada_venta,Precio_litro_gasoleoB,Precio_litro_Fitosanitario,Precio_Abono_kg,Gastos_fijos_por_ha,Descuento_humedad_por_t
Cultivo,,,,,,
Arroz,300.00,0.876,100.0,0.50,550.00,-25.6890
Maiz,252.00,0.876,72.0,0.60,809.50,-20.4652
Tomates,110.52,0.876,127.0,0.65,2154.22,NaN


In [121]:
df["Precio_unitario_semilla"] = (
    df["Precio_Semillas_ha"] /
    df["Semillas_ha"]
)

In [122]:
precio_semilla_por_cultivo = (
    df.groupby("Cultivo")["Precio_unitario_semilla"]
    .mean()
)

precio_semilla_por_cultivo

Cultivo
Arroz      19.980000
Maiz        0.003654
Tomates     0.059800
Name: Precio_unitario_semilla, dtype: float64

In [123]:
def calcular_resultado_economico(
    cultivo,
    superficie,
    predicciones
):
    
    # --------------------------------
    # Parámetros económicos del cultivo
    # --------------------------------
    
    parametros = parametros_constantes.loc[cultivo]
    
    precio_venta = parametros["Precio_tonelada_venta"]
    precio_gasoil = parametros["Precio_litro_gasoleoB"]
    precio_fitosanitario = parametros["Precio_litro_Fitosanitario"]
    precio_abono = parametros["Precio_Abono_kg"]
    gastos_fijos = parametros["Gastos_fijos_por_ha"]
    
    precio_semilla = precio_semilla_por_cultivo.loc[cultivo]
    
    # --------------------------------
    # Predicciones técnicas
    # --------------------------------
    
    rendimiento = predicciones["Rend_t_ha"]
    consumo = predicciones["Consumo_L_ha"]
    fitosanitarios = predicciones["Fitosanitarios_ha"]
    semillas = predicciones["Semillas_ha"]
    abono = predicciones["Abono_ha"]
    pasadas = predicciones["Pasadas_ha"]
    
    # --------------------------------
    # Ingreso
    # --------------------------------
    
    ingreso_bruto_ha = (
        rendimiento * precio_venta
    )
    
    # --------------------------------
    # Costes
    # --------------------------------
    
    coste_gasoil_ha = (
        consumo
        * precio_gasoil
        * pasadas
    )
    
    coste_fitosanitarios_ha = (
        fitosanitarios
        * precio_fitosanitario
    )
    
    coste_semillas_ha = (
        semillas
        * precio_semilla
    )
    
    coste_abono_ha = (
        abono
        * precio_abono
    )
    
    # --------------------------------
    # Coste de secado por humedad
    # --------------------------------
    
    descuento_humedad = parametros[
        "Descuento_humedad_por_t"
    ]
    
    if pd.isna(descuento_humedad):
        coste_secado_ha = 0
    else:
        coste_secado_ha = (
            -descuento_humedad
            * rendimiento
        )
    
    # --------------------------------
    # Coste total
    # --------------------------------
    
    coste_total_ha = (
        coste_gasoil_ha
        + coste_fitosanitarios_ha
        + coste_semillas_ha
        + coste_abono_ha
        + gastos_fijos
        + coste_secado_ha
    )
    
    # --------------------------------
    # Beneficio
    # --------------------------------
    
    beneficio_ha = (
        ingreso_bruto_ha
        - coste_total_ha
    )
    
    beneficio_total = (
        beneficio_ha
        * superficie
    )
    
    return {
        "Ingreso_bruto_ha": ingreso_bruto_ha,
        "Coste_gasoil_ha": coste_gasoil_ha,
        "Coste_fitosanitarios_ha": coste_fitosanitarios_ha,
        "Coste_semillas_ha": coste_semillas_ha,
        "Coste_abono_ha": coste_abono_ha,
        "Coste_secado_ha": coste_secado_ha,
        "Gastos_fijos_ha": gastos_fijos,
        "Coste_total_ha": coste_total_ha,
        "Beneficio_por_ha": beneficio_ha,
        "Beneficio_total": beneficio_total
    }

In [124]:
#prueba
resultado_tecnico = predecir_resultados(
    cultivo="Maiz",
    RTK=1,
    ISOBUS=1,
    Siembra_variable=0,
    Abono_variable=1,
    Corte_tramos=1
)

In [125]:
superficie = 100

In [126]:
resultado_economico = calcular_resultado_economico(
    cultivo="Maiz",
    superficie=superficie,
    predicciones=resultado_tecnico
)

In [127]:
pd.DataFrame(
    resultado_economico.items(),
    columns=["Indicador", "Valor"]
).round(2)

,Indicador,Valor
0,Ingreso_bruto_ha,4041.00
1,Coste_gasoil_ha,81.03
2,Coste_fitosanitarios_ha,296.71
3,Coste_semillas_ha,364.73
4,Coste_abono_ha,428.56
5,Coste_secado_ha,328.17
6,Gastos_fijos_ha,809.50
7,Coste_total_ha,2308.71
8,Beneficio_por_ha,1732.29
9,Beneficio_total,173229.21


In [128]:
beneficio_historico_configuracion = (
    df[
        (df["Cultivo"] == "Maiz") &
        (df["RTK"] == 1) &
        (df["ISOBUS"] == 1) &
        (df["Siembra_variable"] == 0) &
        (df["Abono_variable"] == 1) &
        (df["Corte_tramos"] == 1)
    ]["Beneficio_por_ha"]
    .mean()
)

beneficio_historico_configuracion

1691.2788084857145

In [129]:
print(
    f"Beneficio observado medio: "
    f"{beneficio_historico_configuracion:.2f} €/ha"
)

print(
    f"Beneficio predicho: "
    f"{resultado_economico['Beneficio_por_ha']:.2f} €/ha"
)

print(
    f"Diferencia: "
    f"{resultado_economico['Beneficio_por_ha'] - beneficio_historico_configuracion:.2f} €/ha"
)

Beneficio observado medio: 1691.28 €/ha
Beneficio predicho: 1732.29 €/ha
Diferencia: 41.01 €/ha


## Evaluación de una inversión tecnológica

El siguiente paso consiste en comparar la configuración tecnológica actual de una explotación con una configuración objetivo.

Para ambas situaciones se estiman los resultados técnicos y económicos. La diferencia entre sus beneficios permite obtener el beneficio incremental esperado asociado a la inversión.

Posteriormente, este beneficio incremental se combina con el coste de adquisición de las nuevas tecnologías para calcular ROI, Payback, VAN y TIR.

De esta forma, AgroInvest evalúa la inversión teniendo en cuenta tanto sus efectos técnicos esperados como su coste económico.

In [131]:
costes_tecnologia = {
    "RTK": 7000,
    "ISOBUS": 2500,
    "Siembra_variable": 35000,
    "Abono_variable": 25000,
    "Corte_tramos": 62000
}

In [132]:
#calcular la inversión necesaria
def calcular_inversion_configuracion(
    configuracion_actual,
    configuracion_objetivo
):
    
    inversion_total = 0
    tecnologias_nuevas = []

    for tecnologia in costes_tecnologia:

        actual = configuracion_actual[tecnologia]
        objetivo = configuracion_objetivo[tecnologia]

        if actual == 0 and objetivo == 1:

            inversion_total += costes_tecnologia[tecnologia]
            tecnologias_nuevas.append(tecnologia)

    return inversion_total, tecnologias_nuevas

In [133]:
def comparar_configuraciones(
    cultivo,
    superficie,
    configuracion_actual,
    configuracion_objetivo
):

    # ----------------------------------
    # Situación actual
    # ----------------------------------

    pred_actual = predecir_resultados(
        cultivo=cultivo,
        **configuracion_actual
    )

    eco_actual = calcular_resultado_economico(
        cultivo=cultivo,
        superficie=superficie,
        predicciones=pred_actual
    )

    # ----------------------------------
    # Situación objetivo
    # ----------------------------------

    pred_objetivo = predecir_resultados(
        cultivo=cultivo,
        **configuracion_objetivo
    )

    eco_objetivo = calcular_resultado_economico(
        cultivo=cultivo,
        superficie=superficie,
        predicciones=pred_objetivo
    )

    # ----------------------------------
    # Beneficio incremental
    # ----------------------------------

    beneficio_incremental_ha = (
        eco_objetivo["Beneficio_por_ha"]
        - eco_actual["Beneficio_por_ha"]
    )

    beneficio_incremental_anual = (
        beneficio_incremental_ha
        * superficie
    )

    # ----------------------------------
    # Inversión
    # ----------------------------------

    inversion, tecnologias_nuevas = (
        calcular_inversion_configuracion(
            configuracion_actual,
            configuracion_objetivo
        )
    )

    return {
        "Beneficio_actual_ha":
            eco_actual["Beneficio_por_ha"],

        "Beneficio_objetivo_ha":
            eco_objetivo["Beneficio_por_ha"],

        "Beneficio_incremental_ha":
            beneficio_incremental_ha,

        "Beneficio_incremental_anual":
            beneficio_incremental_anual,

        "Inversion":
            inversion,

        "Tecnologias_nuevas":
            tecnologias_nuevas
    }

In [134]:
#prueba
configuracion_actual = {
    "RTK": 0,
    "ISOBUS": 0,
    "Siembra_variable": 0,
    "Abono_variable": 0,
    "Corte_tramos": 0
}

In [135]:
configuracion_objetivo = {
    "RTK": 1,
    "ISOBUS": 1,
    "Siembra_variable": 0,
    "Abono_variable": 1,
    "Corte_tramos": 1
}

In [136]:
comparacion = comparar_configuraciones(
    cultivo="Maiz",
    superficie=100,
    configuracion_actual=configuracion_actual,
    configuracion_objetivo=configuracion_objetivo
)

comparacion

{'Beneficio_actual_ha': 1090.5263836303034,
 'Beneficio_objetivo_ha': 1732.2920822330248,
 'Beneficio_incremental_ha': 641.7656986027214,
 'Beneficio_incremental_anual': 64176.56986027214,
 'Inversion': 96500,
 'Tecnologias_nuevas': ['RTK', 'ISOBUS', 'Abono_variable', 'Corte_tramos']}

In [137]:
for clave, valor in comparacion.items():
    print(clave, ":", valor)

Beneficio_actual_ha : 1090.5263836303034
Beneficio_objetivo_ha : 1732.2920822330248
Beneficio_incremental_ha : 641.7656986027214
Beneficio_incremental_anual : 64176.56986027214
Inversion : 96500
Tecnologias_nuevas : ['RTK', 'ISOBUS', 'Abono_variable', 'Corte_tramos']


In [138]:
def calcular_roi(beneficio_anual, inversion):
    if inversion == 0:
        return np.nan

    return (beneficio_anual / inversion) * 100


def calcular_payback(beneficio_anual, inversion):
    if beneficio_anual <= 0:
        return np.inf

    return inversion / beneficio_anual


def calcular_van(
    inversion,
    beneficio_anual,
    vida_util=15,
    tasa_descuento=0.05
):
    flujos = [-inversion] + [beneficio_anual] * vida_util
    return npf.npv(tasa_descuento, flujos)


def calcular_tir(
    inversion,
    beneficio_anual,
    vida_util=15
):
    flujos = [-inversion] + [beneficio_anual] * vida_util
    return npf.irr(flujos) * 100

In [139]:
def evaluar_inversion(
    cultivo,
    superficie,
    configuracion_actual,
    configuracion_objetivo,
    vida_util=15,
    tasa_descuento=0.05
):

    resultado = comparar_configuraciones(
        cultivo=cultivo,
        superficie=superficie,
        configuracion_actual=configuracion_actual,
        configuracion_objetivo=configuracion_objetivo
    )

    inversion = resultado["Inversion"]

    beneficio_anual = (
        resultado["Beneficio_incremental_anual"]
    )

    resultado["ROI"] = calcular_roi(
        beneficio_anual,
        inversion
    )

    resultado["Payback"] = calcular_payback(
        beneficio_anual,
        inversion
    )

    resultado["VAN"] = calcular_van(
        inversion,
        beneficio_anual,
        vida_util,
        tasa_descuento
    )

    resultado["TIR"] = calcular_tir(
        inversion,
        beneficio_anual,
        vida_util
    )

    return resultado

In [140]:
resultado_inversion = evaluar_inversion(
    cultivo="Maiz",
    superficie=100,
    configuracion_actual=configuracion_actual,
    configuracion_objetivo=configuracion_objetivo,
    vida_util=15,
    tasa_descuento=0.05
)

resultado_inversion

{'Beneficio_actual_ha': 1090.5263836303034,
 'Beneficio_objetivo_ha': 1732.2920822330248,
 'Beneficio_incremental_ha': 641.7656986027214,
 'Beneficio_incremental_anual': 64176.56986027214,
 'Inversion': 96500,
 'Tecnologias_nuevas': ['RTK', 'ISOBUS', 'Abono_variable', 'Corte_tramos'],
 'ROI': 66.5042174717846,
 'Payback': 1.5036640351783175,
 'VAN': 569630.8492130319,
 'TIR': 66.47239633280249}

## Evaluación automática de alternativas tecnológicas

Una vez desarrollado el motor predictivo y financiero, AgroInvest puede evaluar automáticamente diferentes configuraciones tecnológicas a partir de la situación actual de una explotación.

Para cada alternativa se estiman:

- tecnologías adicionales necesarias;
- inversión inicial;
- beneficio incremental por hectárea;
- beneficio incremental anual;
- ROI;
- Payback;
- VAN;
- TIR.

Las alternativas se comparan posteriormente para identificar las configuraciones económicamente más atractivas bajo las hipótesis seleccionadas.

In [142]:
from itertools import product

In [143]:
tecnologias_evaluables = [
    "RTK",
    "ISOBUS",
    "Siembra_variable",
    "Abono_variable",
    "Corte_tramos"
]

In [144]:
def generar_configuraciones_objetivo(configuracion_actual):

    configuraciones = []

    for valores in product(
        [0, 1],
        repeat=len(tecnologias_evaluables)
    ):

        configuracion = dict(
            zip(tecnologias_evaluables, valores)
        )

        # -----------------------------------------
        # 1. No eliminar tecnologías ya instaladas
        # -----------------------------------------
        valida = True

        for tecnologia in tecnologias_evaluables:

            if (
                configuracion_actual[tecnologia] == 1
                and configuracion[tecnologia] == 0
            ):
                valida = False
                break

        if not valida:
            continue

        # -----------------------------------------
        # 2. RTK es requisito para las tecnologías
        #    avanzadas
        # -----------------------------------------

        tecnologias_avanzadas = [
            "ISOBUS",
            "Siembra_variable",
            "Abono_variable",
            "Corte_tramos"
        ]

        hay_tecnologia_avanzada = any(
            configuracion[tecnologia] == 1
            for tecnologia in tecnologias_avanzadas
        )

        if (
            hay_tecnologia_avanzada
            and configuracion["RTK"] == 0
        ):
            continue

        # -----------------------------------------
        # 3. Excluir la situación actual
        # -----------------------------------------

        if configuracion == configuracion_actual:
            continue

        configuraciones.append(configuracion)

    return configuraciones

In [145]:
def evaluar_todas_alternativas(
    cultivo,
    superficie,
    configuracion_actual,
    vida_util=15,
    tasa_descuento=0.05
):

    alternativas = generar_configuraciones_objetivo(
        configuracion_actual
    )

    resultados = []

    for configuracion_objetivo in alternativas:

        resultado = evaluar_inversion(
            cultivo=cultivo,
            superficie=superficie,
            configuracion_actual=configuracion_actual,
            configuracion_objetivo=configuracion_objetivo,
            vida_util=vida_util,
            tasa_descuento=tasa_descuento
        )

        resultados.append({
            "Configuracion_objetivo":
                configuracion_objetivo,

            "Tecnologias_nuevas":
                ", ".join(
                    resultado["Tecnologias_nuevas"]
                ),

            "Inversion":
                resultado["Inversion"],

            "Beneficio_actual_ha":
                resultado["Beneficio_actual_ha"],

            "Beneficio_objetivo_ha":
                resultado["Beneficio_objetivo_ha"],

            "Beneficio_incremental_ha":
                resultado["Beneficio_incremental_ha"],

            "Beneficio_incremental_anual":
                resultado["Beneficio_incremental_anual"],

            "ROI":
                resultado["ROI"],

            "Payback":
                resultado["Payback"],

            "VAN":
                resultado["VAN"],

            "TIR":
                resultado["TIR"]
        })

    return pd.DataFrame(resultados)

In [146]:
#probamos
configuracion_actual = {
    "RTK": 0,
    "ISOBUS": 0,
    "Siembra_variable": 0,
    "Abono_variable": 0,
    "Corte_tramos": 0
}

In [147]:
tabla_alternativas = evaluar_todas_alternativas(
    cultivo="Maiz",
    superficie=100,
    configuracion_actual=configuracion_actual,
    vida_util=15,
    tasa_descuento=0.05
)

In [148]:
tabla_alternativas.shape

(16, 11)

In [149]:
ranking_van = (
    tabla_alternativas
    .sort_values(
        "VAN",
        ascending=False
    )
    .reset_index(drop=True)
)

ranking_van[
    [
        "Tecnologias_nuevas",
        "Inversion",
        "Beneficio_incremental_ha",
        "ROI",
        "Payback",
        "VAN",
        "TIR"
    ]
].head(10).round(2)

,Tecnologias_nuevas,Inversion,Beneficio_incremental_ha,ROI,Payback,VAN,TIR
0,"RTK, Siembra_variable, Abono_variable, Corte_t...",129000,908.38,70.42,1.42,813864.41,70.39
1,"RTK, ISOBUS, Siembra_variable, Abono_variable,...",131500,876.15,66.63,1.50,777917.53,66.60
2,"RTK, Abono_variable, Corte_tramos",94000,673.89,71.69,1.39,605476.64,71.67
3,"RTK, ISOBUS, Abono_variable, Corte_tramos",96500,641.77,66.50,1.50,569630.85,66.47
4,"RTK, Siembra_variable, Corte_tramos",104000,587.62,56.50,1.77,505933.52,56.43
5,"RTK, Siembra_variable, Abono_variable",67000,523.25,78.10,1.28,476113.11,78.08
6,"RTK, ISOBUS, Siembra_variable, Corte_tramos",106500,555.41,52.15,1.92,469996.32,52.05
7,"RTK, ISOBUS, Siembra_variable, Abono_variable",69500,491.02,70.65,1.42,440157.20,70.63
8,"RTK, Corte_tramos",69000,353.11,51.18,1.95,297516.28,51.07
9,"RTK, Abono_variable",32000,288.72,90.23,1.11,267685.33,90.22


In [150]:
ranking_payback = (
    tabla_alternativas[
        tabla_alternativas["Payback"] != np.inf
    ]
    .sort_values(
        "Payback",
        ascending=True
    )
)

ranking_payback[
    [
        "Tecnologias_nuevas",
        "Inversion",
        "Beneficio_incremental_ha",
        "ROI",
        "Payback",
        "VAN",
        "TIR"
    ]
].head(10).round(2)

,Tecnologias_nuevas,Inversion,Beneficio_incremental_ha,ROI,Payback,VAN,TIR
2,"RTK, Abono_variable",32000,288.72,90.23,1.11,267685.33,90.22
6,"RTK, Siembra_variable, Abono_variable",67000,523.25,78.10,1.28,476113.11,78.08
10,"RTK, ISOBUS, Abono_variable",34500,256.59,74.37,1.34,231830.51,74.36
3,"RTK, Abono_variable, Corte_tramos",94000,673.89,71.69,1.39,605476.64,71.67
14,"RTK, ISOBUS, Siembra_variable, Abono_variable",69500,491.02,70.65,1.42,440157.20,70.63
7,"RTK, Siembra_variable, Abono_variable, Corte_t...",129000,908.38,70.42,1.42,813864.41,70.39
15,"RTK, ISOBUS, Siembra_variable, Abono_variable,...",131500,876.15,66.63,1.50,777917.53,66.60
11,"RTK, ISOBUS, Abono_variable, Corte_tramos",96500,641.77,66.50,1.50,569630.85,66.47
5,"RTK, Siembra_variable, Corte_tramos",104000,587.62,56.50,1.77,505933.52,56.43
13,"RTK, ISOBUS, Siembra_variable, Corte_tramos",106500,555.41,52.15,1.92,469996.32,52.05


In [151]:
alternativas_rentables = tabla_alternativas[
    tabla_alternativas["Beneficio_incremental_anual"] > 0
].copy()

In [152]:
alternativas_rentables.sort_values(
    "VAN",
    ascending=False
).head(10).round(2)

,Configuracion_objetivo,Tecnologias_nuevas,Inversion,Beneficio_actual_ha,Beneficio_objetivo_ha,Beneficio_incremental_ha,Beneficio_incremental_anual,ROI,Payback,VAN,TIR
7,"{'RTK': 1, 'ISOBUS': 0, 'Siembra_variable': 1,...","RTK, Siembra_variable, Abono_variable, Corte_t...",129000,1090.53,1998.90,908.38,90837.71,70.42,1.42,813864.41,70.39
15,"{'RTK': 1, 'ISOBUS': 1, 'Siembra_variable': 1,...","RTK, ISOBUS, Siembra_variable, Abono_variable,...",131500,1090.53,1966.68,876.15,87615.37,66.63,1.50,777917.53,66.60
3,"{'RTK': 1, 'ISOBUS': 0, 'Siembra_variable': 0,...","RTK, Abono_variable, Corte_tramos",94000,1090.53,1764.42,673.89,67389.18,71.69,1.39,605476.64,71.67
11,"{'RTK': 1, 'ISOBUS': 1, 'Siembra_variable': 0,...","RTK, ISOBUS, Abono_variable, Corte_tramos",96500,1090.53,1732.29,641.77,64176.57,66.50,1.50,569630.85,66.47
5,"{'RTK': 1, 'ISOBUS': 0, 'Siembra_variable': 1,...","RTK, Siembra_variable, Corte_tramos",104000,1090.53,1678.15,587.62,58762.39,56.50,1.77,505933.52,56.43
6,"{'RTK': 1, 'ISOBUS': 0, 'Siembra_variable': 1,...","RTK, Siembra_variable, Abono_variable",67000,1090.53,1613.77,523.25,52324.76,78.10,1.28,476113.11,78.08
13,"{'RTK': 1, 'ISOBUS': 1, 'Siembra_variable': 1,...","RTK, ISOBUS, Siembra_variable, Corte_tramos",106500,1090.53,1645.94,555.41,55540.97,52.15,1.92,469996.32,52.05
14,"{'RTK': 1, 'ISOBUS': 1, 'Siembra_variable': 1,...","RTK, ISOBUS, Siembra_variable, Abono_variable",69500,1090.53,1581.54,491.02,49101.54,70.65,1.42,440157.20,70.63
1,"{'RTK': 1, 'ISOBUS': 0, 'Siembra_variable': 0,...","RTK, Corte_tramos",69000,1090.53,1443.64,353.11,35311.02,51.18,1.95,297516.28,51.07
2,"{'RTK': 1, 'ISOBUS': 0, 'Siembra_variable': 0,...","RTK, Abono_variable",32000,1090.53,1379.25,288.72,28872.37,90.23,1.11,267685.33,90.22


In [153]:
pd.crosstab(
    df["RTK"],
    df["ISOBUS"]
)

ISOBUS,0,1
RTK,,
0,18,0
1,0,87


In [154]:
costes_tecnologia = {
    "RTK_ISOBUS": 9500,
    "Siembra_variable": 35000,
    "Abono_variable": 25000,
    "Corte_tramos": 62000
}

In [155]:
tecnologias_evaluables = [
    "RTK_ISOBUS",
    "Siembra_variable",
    "Abono_variable",
    "Corte_tramos"
]

In [156]:
def expandir_configuracion(configuracion):
    
    return {
        "RTK": configuracion["RTK_ISOBUS"],
        "ISOBUS": configuracion["RTK_ISOBUS"],
        "Siembra_variable": configuracion["Siembra_variable"],
        "Abono_variable": configuracion["Abono_variable"],
        "Corte_tramos": configuracion["Corte_tramos"]
    }

In [157]:
def calcular_inversion_configuracion(
    configuracion_actual,
    configuracion_objetivo
):
    
    inversion_total = 0
    tecnologias_nuevas = []

    for tecnologia in costes_tecnologia:

        actual = configuracion_actual[tecnologia]
        objetivo = configuracion_objetivo[tecnologia]

        if actual == 0 and objetivo == 1:
            inversion_total += costes_tecnologia[tecnologia]
            tecnologias_nuevas.append(tecnologia)

    return inversion_total, tecnologias_nuevas

In [158]:
def generar_configuraciones_objetivo(configuracion_actual):

    configuraciones = []

    for valores in product(
        [0, 1],
        repeat=len(tecnologias_evaluables)
    ):

        configuracion = dict(
            zip(tecnologias_evaluables, valores)
        )

        # -----------------------------------------
        # 1. No eliminar tecnologías ya instaladas
        # -----------------------------------------

        valida = True

        for tecnologia in tecnologias_evaluables:

            if (
                configuracion_actual[tecnologia] == 1
                and configuracion[tecnologia] == 0
            ):
                valida = False
                break

        if not valida:
            continue

        # -----------------------------------------
        # 2. RTK + ISOBUS es requisito previo
        # -----------------------------------------

        tecnologias_avanzadas = [
            "Siembra_variable",
            "Abono_variable",
            "Corte_tramos"
        ]

        hay_tecnologia_avanzada = any(
            configuracion[tecnologia] == 1
            for tecnologia in tecnologias_avanzadas
        )

        if (
            hay_tecnologia_avanzada
            and configuracion["RTK_ISOBUS"] == 0
        ):
            continue

        # -----------------------------------------
        # 3. No evaluar la situación actual
        # -----------------------------------------

        if configuracion == configuracion_actual:
            continue

        configuraciones.append(configuracion)

    return configuraciones

In [159]:
def comparar_configuraciones(
    cultivo,
    superficie,
    configuracion_actual,
    configuracion_objetivo
):

    # ----------------------------------
    # Expandir configuraciones
    # RTK_ISOBUS -> RTK + ISOBUS
    # ----------------------------------

    config_actual_modelo = expandir_configuracion(
        configuracion_actual
    )

    config_objetivo_modelo = expandir_configuracion(
        configuracion_objetivo
    )

    # ----------------------------------
    # Situación actual
    # ----------------------------------

    pred_actual = predecir_resultados(
        cultivo=cultivo,
        **config_actual_modelo
    )

    eco_actual = calcular_resultado_economico(
        cultivo=cultivo,
        superficie=superficie,
        predicciones=pred_actual
    )

    # ----------------------------------
    # Situación objetivo
    # ----------------------------------

    pred_objetivo = predecir_resultados(
        cultivo=cultivo,
        **config_objetivo_modelo
    )

    eco_objetivo = calcular_resultado_economico(
        cultivo=cultivo,
        superficie=superficie,
        predicciones=pred_objetivo
    )

    # ----------------------------------
    # Beneficio incremental
    # ----------------------------------

    beneficio_incremental_ha = (
        eco_objetivo["Beneficio_por_ha"]
        - eco_actual["Beneficio_por_ha"]
    )

    beneficio_incremental_anual = (
        beneficio_incremental_ha
        * superficie
    )

    # ----------------------------------
    # Inversión
    # ----------------------------------

    inversion, tecnologias_nuevas = (
        calcular_inversion_configuracion(
            configuracion_actual,
            configuracion_objetivo
        )
    )

    return {
        "Beneficio_actual_ha":
            eco_actual["Beneficio_por_ha"],

        "Beneficio_objetivo_ha":
            eco_objetivo["Beneficio_por_ha"],

        "Beneficio_incremental_ha":
            beneficio_incremental_ha,

        "Beneficio_incremental_anual":
            beneficio_incremental_anual,

        "Inversion":
            inversion,

        "Tecnologias_nuevas":
            tecnologias_nuevas
    }

In [160]:
configuracion_actual = {
    "RTK_ISOBUS": 0,
    "Siembra_variable": 0,
    "Abono_variable": 0,
    "Corte_tramos": 0
}

In [161]:
configuracion_objetivo = {
    "RTK_ISOBUS": 1,
    "Siembra_variable": 0,
    "Abono_variable": 1,
    "Corte_tramos": 1
}

In [162]:
config_actual_modelo = expandir_configuracion(
    configuracion_actual
)

config_objetivo_modelo = expandir_configuracion(
    configuracion_objetivo
)

In [163]:
config_actual_modelo

{'RTK': 0,
 'ISOBUS': 0,
 'Siembra_variable': 0,
 'Abono_variable': 0,
 'Corte_tramos': 0}

In [164]:
config_objetivo_modelo

{'RTK': 1,
 'ISOBUS': 1,
 'Siembra_variable': 0,
 'Abono_variable': 1,
 'Corte_tramos': 1}

In [165]:
tabla_alternativas = evaluar_todas_alternativas(
    cultivo="Maiz",
    superficie=100,
    configuracion_actual=configuracion_actual,
    vida_util=15,
    tasa_descuento=0.05
)

In [166]:
tabla_alternativas.shape

(8, 11)

In [167]:
ranking_van = (
    tabla_alternativas
    .sort_values(
        "VAN",
        ascending=False
    )
    .reset_index(drop=True)
)

In [168]:
ranking_van[
    [
        "Tecnologias_nuevas",
        "Inversion",
        "Beneficio_incremental_ha",
        "ROI",
        "Payback",
        "VAN",
        "TIR"
    ]
].head(10).round(2)

,Tecnologias_nuevas,Inversion,Beneficio_incremental_ha,ROI,Payback,VAN,TIR
0,"RTK_ISOBUS, Siembra_variable, Abono_variable, ...",131500,876.15,66.63,1.50,777917.53,66.60
1,"RTK_ISOBUS, Abono_variable, Corte_tramos",96500,641.77,66.50,1.50,569630.85,66.47
2,"RTK_ISOBUS, Siembra_variable, Corte_tramos",106500,555.41,52.15,1.92,469996.32,52.05
3,"RTK_ISOBUS, Siembra_variable, Abono_variable",69500,491.02,70.65,1.42,440157.20,70.63
4,"RTK_ISOBUS, Corte_tramos",71500,320.99,44.89,2.23,261680.18,44.72
5,"RTK_ISOBUS, Abono_variable",34500,256.59,74.37,1.34,231830.51,74.36
6,"RTK_ISOBUS, Siembra_variable",44500,170.28,38.26,2.61,132243.77,37.96
7,RTK_ISOBUS,9500,-64.18,-67.55,inf,-76112.38,NaN


## Escenarios económicos

AgroInvest permite evaluar una misma configuración tecnológica bajo diferentes condiciones económicas.

Se consideran tres escenarios:

- **Precios constantes:** utiliza precios medios normalizados y permite comparar alternativas reduciendo el efecto de la volatilidad del mercado.
- **Precios históricos:** utiliza los precios reales registrados en una campaña determinada.
- **Precios personalizados:** permite al usuario introducir sus propias hipótesis de precios y costes.

El modelo técnico permanece constante en los tres casos. Únicamente cambian los parámetros económicos utilizados para transformar las predicciones técnicas en ingresos, costes y beneficios.

In [170]:
df_historicos = pd.read_excel(
    "Balance_economico_precios_distintos.xlsx"
)

df_historicos.columns = df_historicos.columns.str.strip()

df_historicos["Parcela"] = (
    df_historicos["Parcela"]
    .astype(str)
    .str.strip()
)

df_historicos["Cultivo"] = (
    df_historicos["Cultivo"]
    .astype(str)
    .str.strip()
)

In [171]:
#coste unitario de semilla
df["Precio_unitario_semilla"] = (
    df["Precio_Semillas_ha"]
    / df["Semillas_ha"]
)

df_historicos["Precio_unitario_semilla"] = (
    df_historicos["Precio_Semillas_ha"]
    / df_historicos["Semillas_ha"]
)

In [172]:
parametros_constantes = (
    df.groupby("Cultivo")
    .agg(
        Precio_tonelada_venta=("Precio_tonelada_venta", "mean"),
        Precio_litro_gasoleoB=("Precio_litro_gasoleoB", "mean"),
        Precio_litro_Fitosanitario=("Precio_litro_Fitosanitario", "mean"),
        Precio_unitario_semilla=("Precio_unitario_semilla", "mean"),
        Precio_Abono_kg=("Precio_Abono_kg", "mean"),
        Gastos_fijos_por_ha=("Gastos_fijos_por_ha", "mean"),
        Descuento_humedad_por_t=("Descuento_humedad_por_t", "mean")
    )
)

parametros_constantes.round(4)

,Precio_tonelada_venta,Precio_litro_gasoleoB,Precio_litro_Fitosanitario,Precio_unitario_semilla,Precio_Abono_kg,Gastos_fijos_por_ha,Descuento_humedad_por_t
Cultivo,,,,,,,
Arroz,300.00,0.876,100.0,19.9800,0.50,550.00,-25.6890
Maiz,252.00,0.876,72.0,0.0037,0.60,809.50,-20.4652
Tomates,110.52,0.876,127.0,0.0598,0.65,2154.22,NaN


In [173]:
parametros_historicos = (
    df_historicos
    .groupby(["Cultivo", "Año"])
    .agg(
        Precio_tonelada_venta=("Precio_tonelada_venta", "mean"),
        Precio_litro_gasoleoB=("Precio_litro_gasoleoB", "mean"),
        Precio_litro_Fitosanitario=("Precio_litro_Fitosanitario", "mean"),
        Precio_unitario_semilla=("Precio_unitario_semilla", "mean"),
        Precio_Abono_kg=("Precio_Abono_kg", "mean"),
        Gastos_fijos_por_ha=("Gastos_fijos_por_ha", "mean"),
        Descuento_humedad_por_t=("Descuento_humedad_por_t", "mean")
    )
)

parametros_historicos.round(4)

Precio_tonelada_venta  Precio_litro_gasoleoB  \
Cultivo Año                                                  
Arroz   2020                  320.0                 0.6010   
        2021                  330.0                 0.7101   
        2023                  550.0                 0.7704   
        2024                  440.0                 1.0247   
        2025                  280.0                 0.9621   
        2026                  300.0                 1.2701   
Maiz    2020                  220.0                 0.6010   
        2021                  260.0                 0.7101   
        2023                  290.0                 0.7704   
        2024                  252.0                 1.0247   
        2025                  245.0                 0.9621   
        2026                  250.0                 1.2701   
Tomates 2020                   75.0                 0.6010   
        2021                   80.0                 0.7101   
        2023                  150.0                 0.7704   
        2024                  125.0                 1.0247   
        2025                  105.0                 0.9621   
        2026                  107.0                 1.2701   

              Precio_litro_Fitosanitario  Precio_unitario_semilla  \
Cultivo Año                                                         
Arroz   2020                        86.0                  18.1000   
        2021                        89.0                  21.2000   
        2023                        94.0                  20.2000   
        2024                       100.0                  19.0000   
        2025                       103.0                  18.9000   
        2026                       108.0                  18.0000   
Maiz    2020                        63.0                   0.0030   
        2021                        72.0                   0.0031   
        2023                        70.0                   0.0034   
        2024                        72.0                   0.0037   
        2025                        74.0                   0.0038   
        2026                        78.0                   0.0038   
Tomates 2020                       110.0                   0.0500   
        2021                       115.0                   0.0500   
        2023                       120.0                   0.0550   
        2024                       127.0                   0.0598   
        2025                       130.0                   0.0600   
        2026                       132.0                   0.0630   

              Precio_Abono_kg  Gastos_fijos_por_ha  Descuento_humedad_por_t  
Cultivo Año                                                                  
Arroz   2020             0.38               420.00                 -27.7437  
        2021             0.42               480.00                 -25.2780  
        2023             0.65               530.00                 -25.2780  
        2024             0.50               550.00                 -25.2780  
        2025             0.48               570.00                 -25.2780  
        2026             0.53               590.00                 -25.2780  
Maiz    2020             0.42               669.50                 -24.6186  
        2021             0.50               709.50                 -20.9589  
        2023             0.70               789.50                 -23.7100  
        2024             0.57               809.50                 -21.0773  
        2025             0.58               829.50                 -22.3668  
        2026             0.64               869.50                 -10.4000  
Tomates 2020             0.50              1874.22                      NaN  
        2021             0.60              2008.22                      NaN  
        2023             0.77              2044.22                      NaN  
        2024             0.62              2154.22                

In [174]:
def obtener_parametros_economicos(
    cultivo,
    escenario="constantes",
    anio=None,
    personalizados=None
):

    if escenario == "constantes":

        return (
            parametros_constantes
            .loc[cultivo]
            .to_dict()
        )

    elif escenario == "historicos":

        if anio is None:
            raise ValueError(
                "Debe indicarse un año para el escenario histórico."
            )

        return (
            parametros_historicos
            .loc[(cultivo, anio)]
            .to_dict()
        )

    elif escenario == "personalizados":

        if personalizados is None:
            raise ValueError(
                "Debe proporcionar parámetros personalizados."
            )

        return personalizados

    else:
        raise ValueError("Escenario económico no reconocido.")

In [175]:
obtener_parametros_economicos(
    cultivo="Maiz",
    escenario="historicos",
    anio=2024
)

{'Precio_tonelada_venta': 252.0,
 'Precio_litro_gasoleoB': 1.0247,
 'Precio_litro_Fitosanitario': 72.0,
 'Precio_unitario_semilla': 0.003654,
 'Precio_Abono_kg': 0.57,
 'Gastos_fijos_por_ha': 809.5,
 'Descuento_humedad_por_t': -21.077285714285715}

In [176]:
def calcular_resultado_economico(
    cultivo,
    superficie,
    predicciones,
    parametros
):

    precio_venta = parametros["Precio_tonelada_venta"]
    precio_gasoil = parametros["Precio_litro_gasoleoB"]
    precio_fitosanitario = parametros["Precio_litro_Fitosanitario"]
    precio_semilla = parametros["Precio_unitario_semilla"]
    precio_abono = parametros["Precio_Abono_kg"]
    gastos_fijos = parametros["Gastos_fijos_por_ha"]
    descuento_humedad = parametros["Descuento_humedad_por_t"]

    rendimiento = predicciones["Rend_t_ha"]
    consumo = predicciones["Consumo_L_ha"]
    fitosanitarios = predicciones["Fitosanitarios_ha"]
    semillas = predicciones["Semillas_ha"]
    abono = predicciones["Abono_ha"]
    pasadas = predicciones["Pasadas_ha"]

    ingreso_bruto_ha = (
        rendimiento * precio_venta
    )

    coste_gasoil_ha = (
        consumo
        * precio_gasoil
        * pasadas
    )

    coste_fitosanitarios_ha = (
        fitosanitarios
        * precio_fitosanitario
    )

    coste_semillas_ha = (
        semillas
        * precio_semilla
    )

    coste_abono_ha = (
        abono
        * precio_abono
    )

    if pd.isna(descuento_humedad):
        coste_secado_ha = 0
    else:
        coste_secado_ha = (
            -descuento_humedad
            * rendimiento
        )

    coste_total_ha = (
        coste_gasoil_ha
        + coste_fitosanitarios_ha
        + coste_semillas_ha
        + coste_abono_ha
        + gastos_fijos
        + coste_secado_ha
    )

    beneficio_ha = (
        ingreso_bruto_ha
        - coste_total_ha
    )

    beneficio_total = (
        beneficio_ha * superficie
    )

    return {
        "Ingreso_bruto_ha": ingreso_bruto_ha,
        "Coste_gasoil_ha": coste_gasoil_ha,
        "Coste_fitosanitarios_ha": coste_fitosanitarios_ha,
        "Coste_semillas_ha": coste_semillas_ha,
        "Coste_abono_ha": coste_abono_ha,
        "Coste_secado_ha": coste_secado_ha,
        "Gastos_fijos_ha": gastos_fijos,
        "Coste_total_ha": coste_total_ha,
        "Beneficio_por_ha": beneficio_ha,
        "Beneficio_total": beneficio_total
    }

In [177]:
def comparar_configuraciones(
    cultivo,
    superficie,
    configuracion_actual,
    configuracion_objetivo,
    parametros
):

    # ----------------------------------
    # Expandir configuraciones
    # RTK_ISOBUS -> RTK + ISOBUS
    # ----------------------------------

    config_actual_modelo = expandir_configuracion(
        configuracion_actual
    )

    config_objetivo_modelo = expandir_configuracion(
        configuracion_objetivo
    )

    # ----------------------------------
    # Situación actual
    # ----------------------------------

    pred_actual = predecir_resultados(
        cultivo=cultivo,
        **config_actual_modelo
    )

    eco_actual = calcular_resultado_economico(
        cultivo=cultivo,
        superficie=superficie,
        predicciones=pred_actual,
        parametros=parametros
    )

    # ----------------------------------
    # Situación objetivo
    # ----------------------------------

    pred_objetivo = predecir_resultados(
        cultivo=cultivo,
        **config_objetivo_modelo
    )

    eco_objetivo = calcular_resultado_economico(
        cultivo=cultivo,
        superficie=superficie,
        predicciones=pred_objetivo,
        parametros=parametros
    )

    # ----------------------------------
    # Beneficio incremental
    # ----------------------------------

    beneficio_incremental_ha = (
        eco_objetivo["Beneficio_por_ha"]
        - eco_actual["Beneficio_por_ha"]
    )

    beneficio_incremental_anual = (
        beneficio_incremental_ha
        * superficie
    )

    # ----------------------------------
    # Inversión necesaria
    # ----------------------------------

    inversion, tecnologias_nuevas = (
        calcular_inversion_configuracion(
            configuracion_actual,
            configuracion_objetivo
        )
    )

    return {
        "Beneficio_actual_ha":
            eco_actual["Beneficio_por_ha"],

        "Beneficio_objetivo_ha":
            eco_objetivo["Beneficio_por_ha"],

        "Beneficio_incremental_ha":
            beneficio_incremental_ha,

        "Beneficio_incremental_anual":
            beneficio_incremental_anual,

        "Inversion":
            inversion,

        "Tecnologias_nuevas":
            tecnologias_nuevas
    }

In [178]:
def evaluar_inversion(
    cultivo,
    superficie,
    configuracion_actual,
    configuracion_objetivo,
    parametros,
    vida_util=15,
    tasa_descuento=0.05
):

    # ----------------------------------
    # Comparación técnica y económica
    # ----------------------------------

    resultado = comparar_configuraciones(
        cultivo=cultivo,
        superficie=superficie,
        configuracion_actual=configuracion_actual,
        configuracion_objetivo=configuracion_objetivo,
        parametros=parametros
    )

    inversion = resultado["Inversion"]

    beneficio_anual = (
        resultado["Beneficio_incremental_anual"]
    )

    # ----------------------------------
    # Indicadores financieros
    # ----------------------------------

    resultado["ROI"] = calcular_roi(
        beneficio_anual,
        inversion
    )

    resultado["Payback"] = calcular_payback(
        beneficio_anual,
        inversion
    )

    resultado["VAN"] = calcular_van(
        inversion,
        beneficio_anual,
        vida_util,
        tasa_descuento
    )

    resultado["TIR"] = calcular_tir(
        inversion,
        beneficio_anual,
        vida_util
    )

    return resultado

In [179]:
def evaluar_todas_alternativas(
    cultivo,
    superficie,
    configuracion_actual,
    parametros,
    vida_util=15,
    tasa_descuento=0.05
):

    # ----------------------------------
    # Generar todas las configuraciones
    # objetivo válidas
    # ----------------------------------

    alternativas = generar_configuraciones_objetivo(
        configuracion_actual
    )

    resultados = []

    # ----------------------------------
    # Evaluar cada alternativa
    # ----------------------------------

    for configuracion_objetivo in alternativas:

        resultado = evaluar_inversion(
            cultivo=cultivo,
            superficie=superficie,
            configuracion_actual=configuracion_actual,
            configuracion_objetivo=configuracion_objetivo,
            parametros=parametros,
            vida_util=vida_util,
            tasa_descuento=tasa_descuento
        )

        resultados.append({
            "Configuracion_objetivo":
                configuracion_objetivo,

            "Tecnologias_nuevas":
                ", ".join(
                    resultado["Tecnologias_nuevas"]
                ),

            "Inversion":
                resultado["Inversion"],

            "Beneficio_actual_ha":
                resultado["Beneficio_actual_ha"],

            "Beneficio_objetivo_ha":
                resultado["Beneficio_objetivo_ha"],

            "Beneficio_incremental_ha":
                resultado["Beneficio_incremental_ha"],

            "Beneficio_incremental_anual":
                resultado["Beneficio_incremental_anual"],

            "ROI":
                resultado["ROI"],

            "Payback":
                resultado["Payback"],

            "VAN":
                resultado["VAN"],

            "TIR":
                resultado["TIR"]
        })

    return pd.DataFrame(resultados)

In [180]:
parametros = obtener_parametros_economicos(
    cultivo="Maiz",
    escenario="constantes"
)

tabla_constantes = evaluar_todas_alternativas(
    cultivo="Maiz",
    superficie=100,
    configuracion_actual=configuracion_actual,
    parametros=parametros
)

tabla_constantes

,Configuracion_objetivo,Tecnologias_nuevas,Inversion,Beneficio_actual_ha,Beneficio_objetivo_ha,Beneficio_incremental_ha,Beneficio_incremental_anual,ROI,Payback,VAN,TIR
0,"{'RTK_ISOBUS': 1, 'Siembra_variable': 0, 'Abon...",RTK_ISOBUS,9500,1090.522181,1026.347021,-64.175160,-6417.515974,-67.552800,inf,-76111.621269,NaN
1,"{'RTK_ISOBUS': 1, 'Siembra_variable': 0, 'Abon...","RTK_ISOBUS, Corte_tramos",71500,1090.522181,1411.516393,320.994212,32099.421167,44.894295,2.227454,261681.014935,44.718779
2,"{'RTK_ISOBUS': 1, 'Siembra_variable': 0, 'Abon...","RTK_ISOBUS, Abono_variable",34500,1090.522181,1347.111870,256.589689,25658.968943,74.373823,1.344559,231831.323242,74.356046
3,"{'RTK_ISOBUS': 1, 'Siembra_variable': 0, 'Abon...","RTK_ISOBUS, Abono_variable, Corte_tramos",96500,1090.522181,1732.288741,641.766560,64176.656030,66.504307,1.503662,569631.743622,66.472486
4,"{'RTK_ISOBUS': 1, 'Siembra_variable': 1, 'Abon...","RTK_ISOBUS, Siembra_variable",44500,1090.522181,1260.801902,170.279721,17027.972140,38.265106,2.613347,132244.527901,37.958503
5,"{'RTK_ISOBUS': 1, 'Siembra_variable': 1, 'Abon...","RTK_ISOBUS, Siembra_variable, Corte_tramos",106500,1090.522181,1645.932729,555.410548,55541.054815,52.151225,1.917500,469997.156060,52.054109
6,"{'RTK_ISOBUS': 1, 'Siembra_variable': 1, 'Abon...","RTK_ISOBUS, Siembra_variable, Abono_variable",69500,1090.522181,1581.538367,491.016186,49101.618573,70.649811,1.415432,440158.009906,70.626454
7,"{'RTK_ISOBUS': 1, 'Siembra_variable': 1, 'Abon...","RTK_ISOBUS, Siembra_variable, Abono_variable, ...",131500,1090.522181,1966.676693,876.154512,87615.451193,66.627720,1.500877,777918.422241,66.596193


In [181]:
#histórico
parametros_2024 = obtener_parametros_economicos(
    cultivo="Maiz",
    escenario="historicos",
    anio=2024
)

parametros_2024

{'Precio_tonelada_venta': 252.0,
 'Precio_litro_gasoleoB': 1.0247,
 'Precio_litro_Fitosanitario': 72.0,
 'Precio_unitario_semilla': 0.003654,
 'Precio_Abono_kg': 0.57,
 'Gastos_fijos_por_ha': 809.5,
 'Descuento_humedad_por_t': -21.077285714285715}

In [182]:
tabla_2024 = evaluar_todas_alternativas(
    cultivo="Maiz",
    superficie=100,
    configuracion_actual=configuracion_actual,
    parametros=parametros_2024,
    vida_util=15,
    tasa_descuento=0.05
)

tabla_2024

,Configuracion_objetivo,Tecnologias_nuevas,Inversion,Beneficio_actual_ha,Beneficio_objetivo_ha,Beneficio_incremental_ha,Beneficio_incremental_anual,ROI,Payback,VAN,TIR
0,"{'RTK_ISOBUS': 1, 'Siembra_variable': 0, 'Abon...",RTK_ISOBUS,9500,1086.861811,1026.329969,-60.531842,-6053.184151,-63.717728,inf,-72329.981532,NaN
1,"{'RTK_ISOBUS': 1, 'Siembra_variable': 0, 'Abon...","RTK_ISOBUS, Corte_tramos",71500,1086.861811,1411.363326,324.501515,32450.151489,45.384827,2.203380,265321.475744,45.216296
2,"{'RTK_ISOBUS': 1, 'Siembra_variable': 0, 'Abon...","RTK_ISOBUS, Abono_variable",34500,1086.861811,1345.107492,258.245682,25824.568164,74.853821,1.335937,233550.186526,74.836652
3,"{'RTK_ISOBUS': 1, 'Siembra_variable': 0, 'Abon...","RTK_ISOBUS, Abono_variable, Corte_tramos",96500,1086.861811,1730.149621,643.287810,64328.781022,66.661949,1.500106,571210.749023,66.630504
4,"{'RTK_ISOBUS': 1, 'Siembra_variable': 1, 'Abon...","RTK_ISOBUS, Siembra_variable",44500,1086.861811,1259.937841,173.076030,17307.603031,38.893490,2.571124,135147.000918,38.602892
5,"{'RTK_ISOBUS': 1, 'Siembra_variable': 1, 'Abon...","RTK_ISOBUS, Siembra_variable, Corte_tramos",106500,1086.861811,1644.926111,558.064301,55806.430068,52.400404,1.908382,472751.660433,52.305209
6,"{'RTK_ISOBUS': 1, 'Siembra_variable': 1, 'Abon...","RTK_ISOBUS, Siembra_variable, Abono_variable",69500,1086.861811,1578.682162,491.820351,49182.035145,70.765518,1.413118,440992.706425,70.742360
7,"{'RTK_ISOBUS': 1, 'Siembra_variable': 1, 'Abon...","RTK_ISOBUS, Siembra_variable, Abono_variable, ...",131500,1086.861811,1963.679205,876.817394,87681.739400,66.678129,1.499742,778606.471160,66.646721


In [183]:
ranking_constantes = (
    tabla_constantes
    .sort_values("VAN", ascending=False)
    .reset_index(drop=True)
)

ranking_2024 = (
    tabla_2024
    .sort_values("VAN", ascending=False)
    .reset_index(drop=True)
)

In [184]:
ranking_2024[
    [
        "Tecnologias_nuevas",
        "Inversion",
        "Beneficio_incremental_ha",
        "ROI",
        "Payback",
        "VAN",
        "TIR"
    ]
].round(2)

,Tecnologias_nuevas,Inversion,Beneficio_incremental_ha,ROI,Payback,VAN,TIR
0,"RTK_ISOBUS, Siembra_variable, Abono_variable, ...",131500,876.82,66.68,1.50,778606.47,66.65
1,"RTK_ISOBUS, Abono_variable, Corte_tramos",96500,643.29,66.66,1.50,571210.75,66.63
2,"RTK_ISOBUS, Siembra_variable, Corte_tramos",106500,558.06,52.40,1.91,472751.66,52.31
3,"RTK_ISOBUS, Siembra_variable, Abono_variable",69500,491.82,70.77,1.41,440992.71,70.74
4,"RTK_ISOBUS, Corte_tramos",71500,324.50,45.38,2.20,265321.48,45.22
5,"RTK_ISOBUS, Abono_variable",34500,258.25,74.85,1.34,233550.19,74.84
6,"RTK_ISOBUS, Siembra_variable",44500,173.08,38.89,2.57,135147.00,38.60
7,RTK_ISOBUS,9500,-60.53,-63.72,inf,-72329.98,NaN


In [185]:
#personalizados
parametros_personalizados = {
    "Precio_tonelada_venta": 280,
    "Precio_litro_gasoleoB": 1.05,
    "Precio_litro_Fitosanitario": 75,
    "Precio_unitario_semilla": 0.0037,
    "Precio_Abono_kg": 0.70,
    "Gastos_fijos_por_ha": 850,
    "Descuento_humedad_por_t": -20
}

In [186]:
parametros_personalizados_maiz = obtener_parametros_economicos(
    cultivo="Maiz",
    escenario="personalizados",
    personalizados=parametros_personalizados
)

In [187]:
tabla_personalizada = evaluar_todas_alternativas(
    cultivo="Maiz",
    superficie=100,
    configuracion_actual=configuracion_actual,
    parametros=parametros_personalizados_maiz,
    vida_util=15,
    tasa_descuento=0.05
)

In [188]:
ranking_personalizado = (
    tabla_personalizada
    .sort_values("VAN", ascending=False)
    .reset_index(drop=True)
)

In [189]:
ranking_personalizado[
    [
        "Tecnologias_nuevas",
        "Inversion",
        "Beneficio_incremental_ha",
        "ROI",
        "Payback",
        "VAN",
        "TIR"
    ]
].round(2)

,Tecnologias_nuevas,Inversion,Beneficio_incremental_ha,ROI,Payback,VAN,TIR
0,"RTK_ISOBUS, Siembra_variable, Abono_variable, ...",131500,983.91,74.82,1.34,889761.12,74.80
1,"RTK_ISOBUS, Abono_variable, Corte_tramos",96500,721.41,74.76,1.34,652295.66,74.74
2,"RTK_ISOBUS, Siembra_variable, Corte_tramos",106500,621.80,58.38,1.71,538904.25,58.33
3,"RTK_ISOBUS, Siembra_variable, Abono_variable",69500,554.20,79.74,1.25,505735.47,79.73
4,"RTK_ISOBUS, Corte_tramos",71500,359.26,50.25,1.99,301403.48,50.13
5,"RTK_ISOBUS, Abono_variable",34500,291.65,84.54,1.18,268222.06,84.53
6,"RTK_ISOBUS, Siembra_variable",44500,192.09,43.17,2.32,154887.93,42.96
7,RTK_ISOBUS,9500,-70.48,-74.19,inf,-82660.80,NaN


In [190]:
def mejor_alternativa_por_van(tabla):

    return (
        tabla
        .sort_values("VAN", ascending=False)
        .iloc[0]
    )


mejor_constantes = mejor_alternativa_por_van(
    tabla_constantes
)

mejor_2024 = mejor_alternativa_por_van(
    tabla_2024
)

mejor_personalizado = mejor_alternativa_por_van(
    tabla_personalizada
)

In [191]:
comparacion_escenarios_final = pd.DataFrame([
    {
        "Escenario": "Precios constantes",
        "Tecnologias": mejor_constantes["Tecnologias_nuevas"],
        "Inversion": mejor_constantes["Inversion"],
        "Beneficio_incremental_ha":
            mejor_constantes["Beneficio_incremental_ha"],
        "ROI": mejor_constantes["ROI"],
        "Payback": mejor_constantes["Payback"],
        "VAN": mejor_constantes["VAN"],
        "TIR": mejor_constantes["TIR"]
    },

    {
        "Escenario": "Histórico 2024",
        "Tecnologias": mejor_2024["Tecnologias_nuevas"],
        "Inversion": mejor_2024["Inversion"],
        "Beneficio_incremental_ha":
            mejor_2024["Beneficio_incremental_ha"],
        "ROI": mejor_2024["ROI"],
        "Payback": mejor_2024["Payback"],
        "VAN": mejor_2024["VAN"],
        "TIR": mejor_2024["TIR"]
    },

    {
        "Escenario": "Personalizado",
        "Tecnologias": mejor_personalizado["Tecnologias_nuevas"],
        "Inversion": mejor_personalizado["Inversion"],
        "Beneficio_incremental_ha":
            mejor_personalizado["Beneficio_incremental_ha"],
        "ROI": mejor_personalizado["ROI"],
        "Payback": mejor_personalizado["Payback"],
        "VAN": mejor_personalizado["VAN"],
        "TIR": mejor_personalizado["TIR"]
    }
])

comparacion_escenarios_final.round(2)

,Escenario,Tecnologias,Inversion,Beneficio_incremental_ha,ROI,Payback,VAN,TIR
0,Precios constantes,"RTK_ISOBUS, Siembra_variable, Abono_variable, ...",131500,876.15,66.63,1.50,777918.42,66.60
1,Histórico 2024,"RTK_ISOBUS, Siembra_variable, Abono_variable, ...",131500,876.82,66.68,1.50,778606.47,66.65
2,Personalizado,"RTK_ISOBUS, Siembra_variable, Abono_variable, ...",131500,983.91,74.82,1.34,889761.12,74.80


## Conclusión de los escenarios económicos

AgroInvest permite evaluar una misma inversión bajo tres contextos económicos diferentes: precios constantes, precios históricos y precios personalizados.

El escenario de precios constantes facilita la comparación estructural entre configuraciones tecnológicas, mientras que el escenario histórico permite reproducir las condiciones económicas de una campaña concreta. El escenario personalizado permite al usuario analizar la inversión bajo sus propias expectativas de precios y costes.

Los resultados muestran que las condiciones económicas pueden modificar de forma relevante el beneficio incremental y los indicadores financieros, aunque determinadas configuraciones tecnológicas mantienen una posición favorable en distintos escenarios.

Esta separación entre modelo técnico y escenario económico permite reutilizar las mismas predicciones productivas bajo diferentes hipótesis de mercado sin necesidad de volver a entrenar los modelos.

## Evaluación de explotaciones con varios cultivos

AgroInvest permite analizar explotaciones que incluyen uno o varios cultivos.

Cada cultivo se evalúa de forma independiente, teniendo en cuenta sus características productivas y económicas. El beneficio incremental esperado se calcula para cada cultivo en función de su superficie y posteriormente se agrega para obtener el resultado total de la explotación.

Los costes de implantación tecnológica se contabilizan una única vez, ya que representan inversiones realizadas a nivel de explotación y no una inversión independiente por cada cultivo.

De esta forma, AgroInvest puede evaluar la rentabilidad global de una inversión teniendo en cuenta la composición real de cultivos de la explotación.

In [194]:
explotacion = {
    "Maiz": 60,
    "Arroz": 30,
    "Tomates": 10
}

In [195]:
superficie_total = sum(
    explotacion.values()
)

superficie_total

100

In [196]:
def obtener_parametros_explotacion(
    explotacion,
    escenario="constantes",
    anio=None,
    personalizados=None
):

    parametros_por_cultivo = {}

    for cultivo in explotacion:

        if escenario == "personalizados":

            parametros_por_cultivo[cultivo] = (
                obtener_parametros_economicos(
                    cultivo=cultivo,
                    escenario="personalizados",
                    personalizados=personalizados[cultivo]
                )
            )

        else:

            parametros_por_cultivo[cultivo] = (
                obtener_parametros_economicos(
                    cultivo=cultivo,
                    escenario=escenario,
                    anio=anio
                )
            )

    return parametros_por_cultivo

In [197]:
def comparar_explotacion_multicultivo(
    explotacion,
    configuracion_actual,
    configuracion_objetivo,
    parametros_por_cultivo
):

    resultados_cultivos = []

    beneficio_incremental_total = 0

    for cultivo, superficie in explotacion.items():

        parametros = (
            parametros_por_cultivo[cultivo]
        )

        resultado = comparar_configuraciones(
            cultivo=cultivo,
            superficie=superficie,
            configuracion_actual=configuracion_actual,
            configuracion_objetivo=configuracion_objetivo,
            parametros=parametros
        )

        beneficio_incremental_total += (
            resultado[
                "Beneficio_incremental_anual"
            ]
        )

        resultados_cultivos.append({
            "Cultivo":
                cultivo,

            "Superficie":
                superficie,

            "Beneficio_actual_ha":
                resultado[
                    "Beneficio_actual_ha"
                ],

            "Beneficio_objetivo_ha":
                resultado[
                    "Beneficio_objetivo_ha"
                ],

            "Beneficio_incremental_ha":
                resultado[
                    "Beneficio_incremental_ha"
                ],

            "Beneficio_incremental_total":
                resultado[
                    "Beneficio_incremental_anual"
                ]
        })

    # La inversión se calcula UNA sola vez
    inversion, tecnologias_nuevas = (
        calcular_inversion_configuracion(
            configuracion_actual,
            configuracion_objetivo
        )
    )

    return (
        pd.DataFrame(resultados_cultivos),
        beneficio_incremental_total,
        inversion,
        tecnologias_nuevas
    )

In [198]:
explotacion = {
    "Maiz": 60,
    "Arroz": 30,
    "Tomates": 10
}

In [199]:
parametros_explotacion = (
    obtener_parametros_explotacion(
        explotacion=explotacion,
        escenario="constantes"
    )
)

In [383]:
configuracion_actual = {
    "RTK_ISOBUS": 0,
    "Siembra_variable": 0,
    "Abono_variable": 0,
    "Corte_tramos": 0
}

In [385]:
configuracion_objetivo = {
    "RTK_ISOBUS": 1,
    "Siembra_variable": 0,
    "Abono_variable": 1,
    "Corte_tramos": 1
}

In [387]:
(
    resultados_cultivos,
    beneficio_incremental_total,
    inversion,
    tecnologias_nuevas
) = comparar_explotacion_multicultivo(
    explotacion=explotacion,
    configuracion_actual=configuracion_actual,
    configuracion_objetivo=configuracion_objetivo,
    parametros_por_cultivo=parametros_explotacion
)

In [389]:
resultados_cultivos.round(2)

,Cultivo,Superficie,Beneficio_actual_ha,Beneficio_objetivo_ha,Beneficio_incremental_ha,Beneficio_incremental_total
0,Maiz,60,1090.52,1732.29,641.77,38505.99
1,Arroz,30,-34354.67,-24710.42,9644.25,289327.57
2,Tomates,10,2604.01,5597.81,2993.80,29938.03


In [391]:
print(
    f"Beneficio incremental total: "
    f"{beneficio_incremental_total:,.2f} €/año"
)

print(
    f"Inversión necesaria: "
    f"{inversion:,.2f} €"
)

print(
    "Tecnologías nuevas:",
    tecnologias_nuevas
)

Beneficio incremental total: 357,771.60 €/año
Inversión necesaria: 96,500.00 €
Tecnologías nuevas: ['RTK_ISOBUS', 'Abono_variable', 'Corte_tramos']


In [393]:
#metricas financieras de toda la explotación
roi_explotacion = calcular_roi(
    beneficio_incremental_total,
    inversion
)

payback_explotacion = calcular_payback(
    beneficio_incremental_total,
    inversion
)

van_explotacion = calcular_van(
    inversion,
    beneficio_incremental_total,
    vida_util=15,
    tasa_descuento=0.05
)

tir_explotacion = calcular_tir(
    inversion,
    beneficio_incremental_total,
    vida_util=15
)

In [395]:
print(
    f"ROI explotación: "
    f"{roi_explotacion:.2f} %"
)

print(
    f"Payback explotación: "
    f"{payback_explotacion:.2f} años"
)

print(
    f"VAN explotación: "
    f"{van_explotacion:,.2f} €"
)

print(
    f"TIR explotación: "
    f"{tir_explotacion:.2f} %"
)

ROI explotación: 370.75 %
Payback explotación: 0.27 años
VAN explotación: 3,617,046.83 €
TIR explotación: 370.75 %


In [397]:
config_modelo_prueba = expandir_configuracion(
    configuracion_objetivo
)

for cultivo in ["Maiz", "Arroz", "Tomates"]:

    pred = predecir_resultados(
        cultivo=cultivo,
        **config_modelo_prueba
    )

    print(f"\n--- {cultivo} ---")

    for variable, valor in pred.items():
        print(f"{variable}: {valor:.2f}")


--- Maiz ---
Productividad_ha_h: 1.84
Consumo_L_ha: 10.17
Fitosanitarios_ha: 4.12
Semillas_ha: 99816.17
Abono_ha: 714.26
Pasadas_ha: 9.09
Rend_t_ha: 16.04

--- Arroz ---
Productividad_ha_h: 1.33
Consumo_L_ha: 16.59
Fitosanitarios_ha: 4.27
Semillas_ha: 1274.69
Abono_ha: 716.22
Pasadas_ha: 10.09
Rend_t_ha: 8.16

--- Tomates ---
Productividad_ha_h: 1.29
Consumo_L_ha: 19.45
Fitosanitarios_ha: 13.18
Semillas_ha: 27217.46
Abono_ha: 1343.07
Pasadas_ha: 14.87
Rend_t_ha: 110.20


In [399]:
variables_revision = [
    "Rend_t_ha",
    "Productividad_ha_h",
    "Consumo_L_ha",
    "Fitosanitarios_ha",
    "Semillas_ha",
    "Abono_ha",
    "Pasadas_ha"
]

rangos_reales = (
    df
    .groupby("Cultivo")[variables_revision]
    .agg(["min", "mean", "max"])
    .round(2)
)

rangos_reales

Rend_t_ha                Productividad_ha_h            Consumo_L_ha  \
              min    mean    max                min  mean  max          min   
Cultivo                                                                       
Arroz         6.9    8.18    9.3                1.1  1.31  1.4        16.43   
Maiz         12.7   16.03   19.0                1.6  1.83  1.9        10.20   
Tomates      86.0  106.45  135.0                0.9  1.23  1.3        18.88   

                     Fitosanitarios_ha  ...        Semillas_ha             \
          mean   max               min  ...    max         min       mean   
Cultivo                                 ...                                 
Arroz    17.29  20.0               4.0  ...   4.75          28      29.24   
Maiz     10.71  12.5               4.0  ...   4.25      100000  100705.88   
Tomates  20.35  27.7              13.5  ...  15.00       26000   26000.00   

                Abono_ha                Pasadas_ha             
            max      min     mean   max        min   mean max  
Cultivo                                                        
Arroz        32      700   714.29   760         10  10.00  10  
Maiz     102000      700   714.12   760          9   9.00   9  
Tomates   26000     1350  1387.50  1500         15  15.17  16  

[3 rows x 21 columns]

In [401]:
objetivos_globales_finales = [
    "Productividad_ha_h",
    "Consumo_L_ha",
    "Fitosanitarios_ha",
    "Abono_ha",
    "Pasadas_ha"
]

In [403]:
modelos_semillas = {}

In [405]:
for cultivo in ["Maiz", "Arroz", "Tomates"]:

    df_cultivo = df[
        df["Cultivo"] == cultivo
    ].copy()

    X_semillas = df_cultivo[
        [
            "RTK",
            "ISOBUS",
            "Siembra_variable",
            "Abono_variable",
            "Corte_tramos"
        ]
    ]

    y_semillas = df_cultivo["Semillas_ha"]

    modelo_semillas = Pipeline(
        steps=[
            (
                "escalado",
                StandardScaler()
            ),
            (
                "modelo",
                Ridge(alpha=1.0)
            )
        ]
    )

    modelo_semillas.fit(
        X_semillas,
        y_semillas
    )

    modelos_semillas[cultivo] = (
        modelo_semillas
    )

In [407]:
modelos_finales = {}

for objetivo in objetivos_globales_finales:

    modelo = Pipeline(
        steps=[
            (
                "preprocesador",
                clone(preprocesador_global_final)
            ),
            (
                "modelo",
                Ridge(alpha=1.0)
            )
        ]
    )

    modelo.fit(
        X_global_final,
        df[objetivo]
    )

    modelos_finales[objetivo] = modelo

modelos_finales["Rend_t_ha"] = (
    modelo_final_rendimiento
)

In [409]:
def predecir_resultados(
    cultivo,
    RTK,
    ISOBUS,
    Siembra_variable,
    Abono_variable,
    Corte_tramos
):

    entrada_global = pd.DataFrame({
        "Cultivo": [cultivo],
        "RTK": [RTK],
        "ISOBUS": [ISOBUS],
        "Siembra_variable": [Siembra_variable],
        "Abono_variable": [Abono_variable],
        "Corte_tramos": [Corte_tramos]
    })

    resultados = {}

    # ----------------------------------
    # Modelos globales
    # ----------------------------------

    for objetivo in objetivos_globales_finales:

        resultados[objetivo] = (
            modelos_finales[objetivo]
            .predict(entrada_global)[0]
        )

    # ----------------------------------
    # Semillas: modelo específico
    # por cultivo
    # ----------------------------------

    entrada_semillas = pd.DataFrame({
        "RTK": [RTK],
        "ISOBUS": [ISOBUS],
        "Siembra_variable": [Siembra_variable],
        "Abono_variable": [Abono_variable],
        "Corte_tramos": [Corte_tramos]
    })

    resultados["Semillas_ha"] = (
        modelos_semillas[cultivo]
        .predict(entrada_semillas)[0]
    )

    # ----------------------------------
    # Rendimiento con interacciones
    # cultivo × tecnología
    # ----------------------------------

    entrada_rendimiento = (
        crear_variables_interaccion_final(
            entrada_global
        )
    )

    entrada_rendimiento = (
        entrada_rendimiento[
            X_interacciones_final.columns
        ]
    )

    resultados["Rend_t_ha"] = (
        modelo_final_rendimiento
        .predict(
            entrada_rendimiento
        )[0]
    )

    return resultados

In [411]:
config_modelo_prueba = expandir_configuracion(
    configuracion_objetivo
)

for cultivo in [
    "Maiz",
    "Arroz",
    "Tomates"
]:

    pred = predecir_resultados(
        cultivo=cultivo,
        **config_modelo_prueba
    )

    print(f"\n--- {cultivo} ---")

    for variable, valor in pred.items():
        print(
            f"{variable}: {valor:.2f}"
        )


--- Maiz ---
Productividad_ha_h: 1.84
Consumo_L_ha: 10.17
Fitosanitarios_ha: 4.12
Abono_ha: 714.26
Pasadas_ha: 9.09
Semillas_ha: 101938.61
Rend_t_ha: 16.04

--- Arroz ---
Productividad_ha_h: 1.33
Consumo_L_ha: 16.59
Fitosanitarios_ha: 4.27
Abono_ha: 716.22
Pasadas_ha: 10.09
Semillas_ha: 31.87
Rend_t_ha: 8.16

--- Tomates ---
Productividad_ha_h: 1.29
Consumo_L_ha: 19.45
Fitosanitarios_ha: 13.18
Abono_ha: 1343.07
Pasadas_ha: 14.87
Semillas_ha: 26000.00
Rend_t_ha: 110.20


In [413]:
(
    resultados_cultivos,
    beneficio_incremental_total,
    inversion,
    tecnologias_nuevas
) = comparar_explotacion_multicultivo(
    explotacion=explotacion,
    configuracion_actual=configuracion_actual,
    configuracion_objetivo=configuracion_objetivo,
    parametros_por_cultivo=parametros_explotacion
)

In [415]:
resultados_cultivos.round(2)

,Cultivo,Superficie,Beneficio_actual_ha,Beneficio_objetivo_ha,Beneficio_incremental_ha,Beneficio_incremental_total
0,Maiz,60,1084.26,1724.53,640.27,38416.11
1,Arroz,30,-110.65,121.22,231.87,6955.97
2,Tomates,10,2704.99,5670.61,2965.62,29656.24


## Ranking de inversiones para explotaciones multicultivo

Una vez calculado el beneficio incremental de cada cultivo, AgroInvest puede evaluar automáticamente todas las configuraciones tecnológicas válidas para el conjunto de la explotación.

El beneficio incremental anual total se obtiene agregando el resultado de cada cultivo según su superficie. El coste de implantación tecnológica se contabiliza una sola vez para toda la explotación.

Posteriormente, cada alternativa se evalúa mediante ROI, Payback, VAN y TIR, permitiendo ordenar las inversiones según distintos criterios económicos.

In [418]:
def evaluar_alternativas_multicultivo(
    explotacion,
    configuracion_actual,
    parametros_por_cultivo,
    vida_util=15,
    tasa_descuento=0.05
):

    alternativas = generar_configuraciones_objetivo(
        configuracion_actual
    )

    resultados = []

    for configuracion_objetivo in alternativas:

        (
            detalle_cultivos,
            beneficio_incremental_total,
            inversion,
            tecnologias_nuevas
        ) = comparar_explotacion_multicultivo(
            explotacion=explotacion,
            configuracion_actual=configuracion_actual,
            configuracion_objetivo=configuracion_objetivo,
            parametros_por_cultivo=parametros_por_cultivo
        )

        roi = calcular_roi(
            beneficio_incremental_total,
            inversion
        )

        payback = calcular_payback(
            beneficio_incremental_total,
            inversion
        )

        van = calcular_van(
            inversion,
            beneficio_incremental_total,
            vida_util=vida_util,
            tasa_descuento=tasa_descuento
        )

        tir = calcular_tir(
            inversion,
            beneficio_incremental_total,
            vida_util=vida_util
        )

        resultados.append({
            "Configuracion_objetivo":
                configuracion_objetivo,

            "Tecnologias_nuevas":
                ", ".join(tecnologias_nuevas),

            "Inversion":
                inversion,

            "Beneficio_incremental_anual":
                beneficio_incremental_total,

            "ROI":
                roi,

            "Payback":
                payback,

            "VAN":
                van,

            "TIR":
                tir
        })

    return pd.DataFrame(resultados)

In [420]:
tabla_multicultivo = evaluar_alternativas_multicultivo(
    explotacion=explotacion,
    configuracion_actual=configuracion_actual,
    parametros_por_cultivo=parametros_explotacion,
    vida_util=15,
    tasa_descuento=0.05
)

In [422]:
ranking_multicultivo = (
    tabla_multicultivo
    .sort_values(
        "VAN",
        ascending=False
    )
    .reset_index(drop=True)
)

In [424]:
ranking_multicultivo[
    [
        "Tecnologias_nuevas",
        "Inversion",
        "Beneficio_incremental_anual",
        "ROI",
        "Payback",
        "VAN",
        "TIR"
    ]
].round(2)

,Tecnologias_nuevas,Inversion,Beneficio_incremental_anual,ROI,Payback,VAN,TIR
0,"RTK_ISOBUS, Siembra_variable, Abono_variable, ...",131500,96275.63,73.21,1.37,867808.15,73.19
1,"RTK_ISOBUS, Abono_variable, Corte_tramos",96500,75028.33,77.75,1.29,682268.40,77.74
2,"RTK_ISOBUS, Siembra_variable, Corte_tramos",106500,67716.54,63.58,1.57,596374.52,63.54
3,"RTK_ISOBUS, Siembra_variable, Abono_variable",69500,59420.44,85.50,1.17,547263.88,85.49
4,"RTK_ISOBUS, Corte_tramos",71500,46466.40,64.99,1.54,410805.30,64.95
5,"RTK_ISOBUS, Abono_variable",34500,38169.28,110.64,0.90,361684.12,110.63
6,"RTK_ISOBUS, Siembra_variable",44500,30862.10,69.35,1.44,275838.04,69.33
7,RTK_ISOBUS,9500,9608.10,101.14,0.99,90228.81,101.14


In [426]:
ranking_multicultivo_payback = (
    tabla_multicultivo[
        tabla_multicultivo["Payback"] != np.inf
    ]
    .sort_values(
        "Payback",
        ascending=True
    )
    .reset_index(drop=True)
)

ranking_multicultivo_payback[
    [
        "Tecnologias_nuevas",
        "Inversion",
        "Beneficio_incremental_anual",
        "ROI",
        "Payback",
        "VAN",
        "TIR"
    ]
].round(2)

,Tecnologias_nuevas,Inversion,Beneficio_incremental_anual,ROI,Payback,VAN,TIR
0,"RTK_ISOBUS, Abono_variable",34500,38169.28,110.64,0.90,361684.12,110.63
1,RTK_ISOBUS,9500,9608.10,101.14,0.99,90228.81,101.14
2,"RTK_ISOBUS, Siembra_variable, Abono_variable",69500,59420.44,85.50,1.17,547263.88,85.49
3,"RTK_ISOBUS, Abono_variable, Corte_tramos",96500,75028.33,77.75,1.29,682268.40,77.74
4,"RTK_ISOBUS, Siembra_variable, Abono_variable, ...",131500,96275.63,73.21,1.37,867808.15,73.19
5,"RTK_ISOBUS, Siembra_variable",44500,30862.10,69.35,1.44,275838.04,69.33
6,"RTK_ISOBUS, Corte_tramos",71500,46466.40,64.99,1.54,410805.30,64.95
7,"RTK_ISOBUS, Siembra_variable, Corte_tramos",106500,67716.54,63.58,1.57,596374.52,63.54


## Conclusión

En este notebook se ha desarrollado el núcleo predictivo y económico de AgroInvest.

El sistema utiliza modelos Ridge para estimar variables técnicas a partir del cultivo y de la configuración tecnológica de la explotación. El rendimiento se modela incorporando interacciones entre cultivo y tecnología, mientras que las semillas se estiman mediante modelos específicos por cultivo debido a las importantes diferencias de escala observadas entre maíz, arroz y tomate.

Las predicciones técnicas se integran posteriormente con un motor económico que permite trabajar con tres escenarios de precios: constantes, históricos y personalizados. A partir de estos parámetros se estiman ingresos, costes y beneficio por hectárea.

AgroInvest permite comparar la situación tecnológica actual con distintas configuraciones objetivo y calcula para cada alternativa el beneficio incremental, la inversión necesaria, el ROI, el Payback, el VAN y la TIR.

El sistema también permite analizar explotaciones con varios cultivos. En este caso, el beneficio incremental se calcula individualmente para cada cultivo según su superficie y posteriormente se agrega, mientras que la inversión tecnológica se contabiliza una sola vez para el conjunto de la explotación.

Los resultados muestran que la alternativa con mayor VAN no tiene por qué coincidir con la alternativa de menor Payback o mayor ROI, por lo que la recomendación final puede adaptarse al criterio económico prioritario del usuario.

Las estimaciones deben interpretarse como apoyo a la decisión y no como garantías de resultado, especialmente en configuraciones tecnológicas o cultivos con menor representación histórica.